In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 10 — SPLIT PROTOCOL EXPERIMENT  (mass, fold 0)
#   Identical model + identical data. ONLY the train/test split changes.
#     A) patient-grouped   (your protocol)
#     B) image-level random (the protocol implied when none is reported)
#   Measures the effect on BOTH segmentation Dice and classification AUC.
#
#   >>> RUN WITH QUICK_TEST = True FIRST <<<
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("HF_HUB_OFFLINE", "1")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score, accuracy_score, balanced_accuracy_score
cv2.setNumThreads(0)

D   = "/root/autodl-tmp/CBIS"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True

QUICK_TEST = False        # <<<<<< set False for the real run
FOLD       = 0
SEED       = 11

SEG_IMG, SEG_BATCH, SEG_EPOCHS, SEG_MULT, SEG_PAT = 256, 16, 25, 4, 6
CLS_IMG, CLS_BATCH, CLS_EPOCHS, CLS_MULT, CLS_PAT = 512, 12, 15, 3, 5
CLS_FREEZE, LR_HEAD, LR_HEAD_FT, LR_BACK, WD, GAMMA = 3, 1e-3, 3e-4, 3e-5, 1e-4, 2.0

if QUICK_TEST:
    SEG_EPOCHS, SEG_MULT, CLS_EPOCHS, CLS_MULT, CLS_FREEZE = 3, 1, 3, 1, 1
    print(">>> QUICK TEST — error check only\n")

d = pd.read_csv(os.path.join(D, "unified_folds_mass.csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_mass")
d["pred"] = d["img"].apply(lambda p: os.path.join(
    PM, os.path.basename(str(p)).replace("_img.png", "") + "_pred.png"))
assert d["pred"].apply(os.path.exists).all(), "predmasks_mass incomplete"

# ══════════════ build the two splits ══════════════
role = d["role_f%d" % FOLD]
G = dict(tr=np.where(role == "train")[0],
         va=np.where(role == "val")[0],
         te=np.where(role == "test")[0])

rs = np.random.RandomState(SEED)
perm = rs.permutation(len(d))
n_tr, n_va = len(G["tr"]), len(G["va"])
L = dict(tr=perm[:n_tr], va=perm[n_tr:n_tr + n_va], te=perm[n_tr + n_va:])

def leak_report(S, name):
    trp, tep = set(d.patient_id[S["tr"]]), set(d.patient_id[S["te"]])
    trl, tel = set(d.lesion_key[S["tr"]]), set(d.lesion_key[S["te"]])
    shared_les = len(trl & tel)
    print("  %-16s train %-5d val %-4d test %-5d | patients in both: %-4d | "
          "test lesions whose other view is in train: %d / %d  (%.0f%%)"
          % (name, len(S["tr"]), len(S["va"]), len(S["te"]),
             len(trp & tep), shared_les, len(tel), 100 * shared_les / max(len(tel), 1)))

print("=" * 96)
print("SPLIT CONSTRUCTION — fold %d, identical sizes" % FOLD)
print("=" * 96)
leak_report(G, "A patient-grouped")
leak_report(L, "B image-level")
print("=" * 96)

# ══════════════ shared caches ══════════════
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
SEG_CACHE, CLS_CACHE = {}, {}
t0 = time.time()
for _, r in d.iterrows():
    k = str(r["img"])
    im = cv2.imread(k, cv2.IMREAD_GRAYSCALE)
    im = np.zeros((512, 512), np.uint8) if im is None else im
    mk = cv2.imread(str(r["msk"]), cv2.IMREAD_GRAYSCALE)
    mk = np.zeros_like(im) if mk is None else mk
    pm = cv2.imread(str(r["pred"]), cv2.IMREAD_GRAYSCALE)
    pm = np.zeros_like(im) if pm is None else pm
    SEG_CACHE[k] = (_clahe.apply(cv2.resize(im, (SEG_IMG, SEG_IMG))),
                    (cv2.resize(mk, (SEG_IMG, SEG_IMG), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
    CLS_CACHE[k] = (_clahe.apply(cv2.resize(im, (CLS_IMG, CLS_IMG))),
                    (cv2.resize(pm, (CLS_IMG, CLS_IMG), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8))
print("cached %d images in %.0fs\n" % (len(SEG_CACHE), time.time() - t0))

MEAN = np.array([0.485, 0.456, 0.406], np.float32).reshape(3, 1, 1)
STD  = np.array([0.229, 0.224, 0.225], np.float32).reshape(3, 1, 1)


def aug_pair(a, b, size):
    if np.random.rand() < .5: a, b = a[:, ::-1], b[:, ::-1]
    if np.random.rand() < .5: a, b = a[::-1, :], b[::-1, :]
    k = np.random.randint(4)
    if k: a, b = np.rot90(a, k), np.rot90(b, k)
    a, b = np.ascontiguousarray(a), np.ascontiguousarray(b)
    if np.random.rand() < .6:
        M = cv2.getRotationMatrix2D((size / 2, size / 2),
                                    np.random.uniform(-25, 25), np.random.uniform(.9, 1.1))
        a = cv2.warpAffine(a, M, (size, size), borderMode=cv2.BORDER_REFLECT)
        b = cv2.warpAffine(b, M, (size, size), flags=cv2.INTER_NEAREST)
    if np.random.rand() < .5:
        a = np.clip(a.astype(np.float32) * np.random.uniform(.85, 1.15), 0, 255).astype(np.uint8)
    return np.ascontiguousarray(a), np.ascontiguousarray(b)


class SegDS(Dataset):
    def __init__(s, idx, aug, mult=1):
        s.idx = np.asarray(idx); s.aug = aug; s.mult = mult if aug else 1
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        im, mk = [a.copy() for a in SEG_CACHE[str(d.iloc[j]["img"])]]
        if s.aug: im, mk = aug_pair(im, mk, SEG_IMG)
        return (torch.from_numpy(im.astype(np.float32) / 255.).unsqueeze(0),
                torch.from_numpy(mk.astype(np.int64)))


class ClsDS(Dataset):
    def __init__(s, idx, aug, mult=1, tta=0):
        s.idx = np.asarray(idx); s.aug = aug; s.mult = mult if aug else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.mult
    def __getitem__(s, i):
        j = int(s.idx[i % len(s.idx)])
        im, mk = [a.copy() for a in CLS_CACHE[str(d.iloc[j]["img"])]]
        if s.aug:
            im, mk = aug_pair(im, mk, CLS_IMG)
        else:
            t = s.tta
            if   t == 1: im, mk = im[:, ::-1], mk[:, ::-1]
            elif t == 2: im, mk = im[::-1, :], mk[::-1, :]
            elif t == 3: im, mk = np.rot90(im, 2), np.rot90(mk, 2)
            im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
        g = im.astype(np.float32) / 255.
        x = ((np.stack([g, g, g], 0) - MEAN) / STD).astype(np.float32)
        return (torch.from_numpy(x), torch.from_numpy(mk.astype(np.float32))[None],
                torch.tensor(int(d.iloc[j]["label"])))


# ══════════════ segmentation model ══════════════
def cb(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))

class AG(nn.Module):
    def __init__(s, g, x, i):
        super().__init__()
        s.Wg = nn.Sequential(nn.Conv2d(g, i, 1), nn.BatchNorm2d(i))
        s.Wx = nn.Sequential(nn.Conv2d(x, i, 1), nn.BatchNorm2d(i))
        s.psi = nn.Sequential(nn.Conv2d(i, 1, 1), nn.BatchNorm2d(1), nn.Sigmoid()); s.r = nn.ReLU(True)
    def forward(s, g, x): return x * s.psi(s.r(s.Wg(g) + s.Wx(x)))

class ASPP(nn.Module):
    def __init__(s, i, o):
        super().__init__()
        s.b0 = nn.Sequential(nn.Conv2d(i, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        s.b1 = nn.Sequential(nn.Conv2d(i, o, 3, padding=6,  dilation=6),  nn.BatchNorm2d(o), nn.ReLU(True))
        s.b2 = nn.Sequential(nn.Conv2d(i, o, 3, padding=12, dilation=12), nn.BatchNorm2d(o), nn.ReLU(True))
        s.b3 = nn.Sequential(nn.Conv2d(i, o, 3, padding=18, dilation=18), nn.BatchNorm2d(o), nn.ReLU(True))
        s.gp = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Conv2d(i, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
        s.proj = nn.Sequential(nn.Conv2d(o * 5, o, 1), nn.BatchNorm2d(o), nn.ReLU(True))
    def forward(s, x):
        g = F.interpolate(s.gp(x), size=x.shape[2:], mode="bilinear", align_corners=False)
        return s.proj(torch.cat([s.b0(x), s.b1(x), s.b2(x), s.b3(x), g], 1))

class DSAttnUNet(nn.Module):
    def __init__(s, b=32):
        super().__init__()
        s.e1, s.e2, s.e3, s.e4 = cb(1, b), cb(b, b*2), cb(b*2, b*4), cb(b*4, b*8)
        s.p = nn.MaxPool2d(2); s.bn = ASPP(b*8, b*16)
        s.u4 = nn.ConvTranspose2d(b*16, b*8, 2, 2); s.a4 = AG(b*8, b*8, b*4); s.d4 = cb(b*16, b*8)
        s.u3 = nn.ConvTranspose2d(b*8,  b*4, 2, 2); s.a3 = AG(b*4, b*4, b*2); s.d3 = cb(b*8,  b*4)
        s.u2 = nn.ConvTranspose2d(b*4,  b*2, 2, 2); s.a2 = AG(b*2, b*2, b);   s.d2 = cb(b*4,  b*2)
        s.u1 = nn.ConvTranspose2d(b*2,  b,   2, 2); s.a1 = AG(b, b, b//2);    s.d1 = cb(b*2,  b)
        s.out = nn.Conv2d(b, 2, 1)
        s.ds2, s.ds3, s.ds4 = nn.Conv2d(b*2, 2, 1), nn.Conv2d(b*4, 2, 1), nn.Conv2d(b*8, 2, 1)
    def forward(s, x):
        e1 = s.e1(x); e2 = s.e2(s.p(e1)); e3 = s.e3(s.p(e2)); e4 = s.e4(s.p(e3))
        bo = s.bn(s.p(e4))
        g4 = s.u4(bo); d4 = s.d4(torch.cat([g4, s.a4(g4, e4)], 1))
        g3 = s.u3(d4); d3 = s.d3(torch.cat([g3, s.a3(g3, e3)], 1))
        g2 = s.u2(d3); d2 = s.d2(torch.cat([g2, s.a2(g2, e2)], 1))
        g1 = s.u1(d2); d1 = s.d1(torch.cat([g1, s.a1(g1, e1)], 1))
        if s.training: return s.out(d1), s.ds2(d2), s.ds3(d3), s.ds4(d4)
        return s.out(d1)


def tversky_ce(lo, t):
    ce = F.cross_entropy(lo.float(), t)
    p = F.softmax(lo.float(), 1)[:, 1]; g = t.float()
    tp = (p * g).sum((1, 2)); fp = (p * (1 - g)).sum((1, 2)); fn = ((1 - p) * g).sum((1, 2))
    return 0.3 * ce + 0.7 * (1 - ((tp + 1) / (tp + 0.7 * fp + 0.3 * fn + 1)).mean())


@torch.no_grad()
def seg_dice(net, idx):
    net.eval(); out = []
    for x, y in DataLoader(SegDS(idx, False), batch_size=SEG_BATCH, shuffle=False, num_workers=0):
        x, y = x.to(DEV), y.to(DEV)
        with torch.amp.autocast(device_type="cuda"):
            o = net(x)
        pr = (torch.softmax(o.float(), 1)[:, 1] > 0.5).float(); gt = y.float()
        tp = (pr * gt).sum((1, 2)); fp = (pr * (1 - gt)).sum((1, 2)); fn = ((1 - pr) * gt).sum((1, 2))
        out += ((2 * tp + 1) / (2 * tp + fp + fn + 1)).cpu().tolist()
    return float(np.mean(out))


def run_seg(S, tag):
    torch.manual_seed(SEED); np.random.seed(SEED)
    net = DSAttnUNet().to(DEV)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    sch = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", patience=4, factor=.5)
    sc = torch.amp.GradScaler()
    tl = DataLoader(SegDS(S["tr"], True, SEG_MULT), batch_size=SEG_BATCH, shuffle=True,
                    num_workers=0, drop_last=True)
    best, bstate, bad = -1, None, 0
    for ep in range(1, SEG_EPOCHS + 1):
        net.train()
        for x, y in tl:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                outs = net(x)
                loss = sum(w * tversky_ce(o, F.interpolate(y[:, None].float(), size=o.shape[2:],
                           mode="nearest")[:, 0].long() if o.shape[2:] != y.shape[1:] else y)
                           for w, o in zip([1.0, .5, .3, .2], outs))
            sc.scale(loss).backward(); sc.step(opt); sc.update()
        v = seg_dice(net, S["va"]); sch.step(v)
        if v > best:
            best, bad = v, 0
            bstate = {q: t.detach().cpu().clone() for q, t in net.state_dict().items()}
        else:
            bad += 1
        print("      [%s] ep %2d  val Dice %.4f" % (tag, ep, v))
        if bad >= SEG_PAT: break
    net.load_state_dict({q: t.to(DEV) for q, t in bstate.items()})
    r = seg_dice(net, S["te"])
    del net; gc.collect(); torch.cuda.empty_cache()
    return r


# ══════════════ classification model ══════════════
class GuidedNet(nn.Module):
    def __init__(s):
        super().__init__()
        try:  dn = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
        except Exception: dn = models.densenet121(weights=None)
        s.b = dn.features
        s.head = nn.Sequential(nn.Linear(2048, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                               nn.Dropout(0.4), nn.Linear(512, 2))
    def forward(s, x, m):
        f = F.relu(s.b(x))
        mm = F.interpolate(m, size=f.shape[2:], mode="bilinear", align_corners=False)
        w = 1.0 + 2.0 * mm
        return s.head(torch.cat([(f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6), f.mean((2, 3))], 1))


@torch.no_grad()
def cls_probs(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0, 1, 2, 3] if tta else [0]):
        ps = []
        for x, m, _ in DataLoader(ClsDS(idx, False, 1, tta=t), batch_size=16,
                                  shuffle=False, num_workers=0):
            x, m = x.to(DEV), m.to(DEV)
            with torch.amp.autocast(device_type="cuda"):
                o = net(x, m)
            ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot + ps
    return tot / (4 if tta else 1)


def run_cls(S, tag):
    torch.manual_seed(SEED); np.random.seed(SEED)
    y = d["label"].values
    n0, n1 = float((y[S["tr"]] == 0).sum()), float((y[S["tr"]] == 1).sum())
    al = torch.tensor([n1 / (n0 + n1), n0 / (n0 + n1)], device=DEV, dtype=torch.float32)
    net = GuidedNet().to(DEV).to(memory_format=torch.channels_last)
    for p in net.b.parameters(): p.requires_grad = False
    hp = [p for n_, p in net.named_parameters() if not n_.startswith("b.")]
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD)
    sch, sc = None, torch.amp.GradScaler()
    tl = DataLoader(ClsDS(S["tr"], True, CLS_MULT), batch_size=CLS_BATCH, shuffle=True,
                    num_workers=0, drop_last=True)
    best, bstate, bad = -1, None, 0
    for ep in range(1, CLS_EPOCHS + 1):
        if ep == CLS_FREEZE + 1:
            for p in net.b.parameters(): p.requires_grad = True
            opt = torch.optim.AdamW([{"params": net.b.parameters(), "lr": LR_BACK},
                                     {"params": hp, "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, CLS_EPOCHS - CLS_FREEZE))
        net.train()
        if ep <= CLS_FREEZE: net.b.eval()
        for x, m, t in tl:
            x = x.to(DEV).to(memory_format=torch.channels_last); m, t = m.to(DEV), t.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o = net(x, m)
                ce = F.cross_entropy(o.float(), t, weight=al, reduction="none")
                loss = ((1 - torch.exp(-ce)) ** GAMMA * ce).mean()
            if not torch.isfinite(loss): continue
            sc.scale(loss).backward(); sc.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0); sc.step(opt); sc.update()
        if sch: sch.step()
        pv = cls_probs(net, S["va"], tta=False)
        a = roc_auc_score(y[S["va"]], pv) if len(set(y[S["va"]])) > 1 else 0
        if a > best:
            best, bad = a, 0
            bstate = {q: t.detach().cpu().clone() for q, t in net.state_dict().items()}
        else:
            bad += 1
        print("      [%s] ep %2d  val AUC %.4f" % (tag, ep, a))
        if bad >= CLS_PAT: break
    net.load_state_dict({q: t.to(DEV) for q, t in bstate.items()})
    p = cls_probs(net, S["te"])
    auc = roc_auc_score(y[S["te"]], p)
    grid = np.linspace(.05, .95, 181)
    thr = grid[int(np.argmax([balanced_accuracy_score(y[S["te"]], (p > t).astype(int)) for t in grid]))]
    acc = accuracy_score(y[S["te"]], (p > thr).astype(int))
    del net; gc.collect(); torch.cuda.empty_cache()
    return auc, acc


# ══════════════ run all four ══════════════
R = {}
for name, S in [("A patient-grouped", G), ("B image-level", L)]:
    print("\n### SEGMENTATION — %s" % name); t0 = time.time()
    R[(name, "dice")] = run_seg(S, name.split()[0])
    print("  TEST Dice %.4f  (%.0fs)" % (R[(name, "dice")], time.time() - t0))

    print("\n### CLASSIFICATION — %s" % name); t0 = time.time()
    auc, acc = run_cls(S, name.split()[0])
    R[(name, "auc")], R[(name, "acc")] = auc, acc
    print("  TEST AUC %.4f  acc %.1f%%  (%.0fs)" % (auc, 100 * acc, time.time() - t0))

print("\n" + "=" * 84)
print("SPLIT PROTOCOL EFFECT — mass, fold %d, identical model and data" % FOLD)
print("=" * 84)
print("%-22s %-16s %-16s %-16s" % ("protocol", "Seg Dice", "Cls AUC", "Cls accuracy"))
print("-" * 84)
for name in ["A patient-grouped", "B image-level"]:
    print("%-22s %-16.4f %-16.4f %-16.1f%%"
          % (name, R[(name, "dice")], R[(name, "auc")], 100 * R[(name, "acc")]))
print("-" * 84)
print("%-22s %+-16.4f %+-16.4f %+-16.1f%%"
      % ("INFLATION",
         R[("B image-level", "dice")] - R[("A patient-grouped", "dice")],
         R[("B image-level", "auc")]  - R[("A patient-grouped", "auc")],
         100 * (R[("B image-level", "acc")] - R[("A patient-grouped", "acc")])))
print("=" * 84)
print("Each CBIS abnormality is imaged in two views. Image-level random splitting")
print("places the same lesion in both training and test sets.")

SPLIT CONSTRUCTION — fold 0, identical sizes
  A patient-grouped train 1202  val 155  test 339   | patients in both: 0    | test lesions whose other view is in train: 0 / 199  (0%)
  B image-level    train 1202  val 155  test 339   | patients in both: 186  | test lesions whose other view is in train: 181 / 302  (60%)
cached 1696 images in 11s


### SEGMENTATION — A patient-grouped
      [A] ep  1  val Dice 0.8632
      [A] ep  2  val Dice 0.8598
      [A] ep  3  val Dice 0.8153
      [A] ep  4  val Dice 0.8634
      [A] ep  5  val Dice 0.8695
      [A] ep  6  val Dice 0.8619
      [A] ep  7  val Dice 0.8518
      [A] ep  8  val Dice 0.8807
      [A] ep  9  val Dice 0.8736
      [A] ep 10  val Dice 0.8810
      [A] ep 11  val Dice 0.8816
      [A] ep 12  val Dice 0.8762
      [A] ep 13  val Dice 0.8728
      [A] ep 14  val Dice 0.8841
      [A] ep 15  val Dice 0.8837
      [A] ep 16  val Dice 0.8929
      [A] ep 17  val Dice 0.8815
      [A] ep 18  val Dice 0.8738
      [A] ep 19  val D

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 14 — IS MORE ENSEMBLING WORTH IT?   no GPU, ~1 minute
#   A) how correlated are your models?
#   B) pairwise ensemble gain
#   C) lesions that EVERY model gets wrong  (the irreducible core)
#   D) oracle ceiling — best possible from combining what you have
# ══════════════════════════════════════════════════════════════════════
import os, itertools, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.metrics import roc_auc_score, accuracy_score
warnings.filterwarnings("ignore")

D = "/root/autodl-tmp/CBIS"
MODELS = [
    ("cv_mass_endtoend.csv",            "prob_pred"),
    ("phase3_mass.csv",                 "prob_pred"),
    ("endtoend_variants_mass.csv",      "e1"),
    ("endtoend_variants_mass.csv",      "e2"),
    ("endtoend_variants_mass.csv",      "e3"),
    ("cv_mass_imageonly_oof.csv",       "prob"),
    ("cv_mass_v2_oof.csv",              "prob"),
    ("cv_mass_efficientnet_b0_oof.csv", "prob"),
    ("cv_mass_convnext_tiny_oof.csv",   "prob"),
    ("cv_mass_resnet50_oof.csv",        "prob"),
]

base = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))[
        ["img", "lesion_key", "label", "fold", "assessment"]].copy()
base["img"] = base["img"].astype(str)
base["label"] = base["label"].astype(int)
base["assessment"] = pd.to_numeric(base["assessment"], errors="coerce")

names = []
for fn, col in MODELS:
    p = os.path.join(D, fn)
    if not os.path.exists(p):
        continue
    x = pd.read_csv(p)
    if "img" not in x.columns or col not in x.columns:
        continue
    t = pd.DataFrame({"img": x["img"].astype(str),
                      "_p": pd.to_numeric(x[col], errors="coerce")}).dropna().drop_duplicates("img")
    m = base.merge(t, on="img", how="left")
    if m["_p"].notna().mean() < 0.98:
        continue
    tag = fn.replace(".csv", "").replace("cv_mass_", "").replace("_oof", "")
    if col.startswith("e") and col[1:].isdigit():
        tag += ":" + col
    base[tag] = m["_p"].values
    names.append(tag)

L = base.groupby("lesion_key").agg({**{n: "mean" for n in names},
                                    "label": "max", "assessment": "max"}).reset_index()
y = L["label"].values
print("models: %d | lesions: %d\n" % (len(names), len(L)))

# ─────────── PART A : correlation ───────────
print("=" * 78); print("PART A — Spearman correlation between models"); print("=" * 78)
C = np.zeros((len(names), len(names)))
for i, a in enumerate(names):
    for j, b in enumerate(names):
        C[i, j] = spearmanr(L[a], L[b]).correlation
hdr = "".join("%8s" % n[:7] for n in names)
print("%-24s%s" % ("", hdr))
for i, a in enumerate(names):
    print("%-24s%s" % (a[:24], "".join("%8.2f" % C[i, j] for j in range(len(names)))))
off = C[np.triu_indices(len(names), 1)]
print("\n  mean off-diagonal correlation: %.2f" % off.mean())
print("  LOW (<0.6) means diverse models -> ensembling should help")
print("  HIGH (>0.8) means they agree -> more of the same will NOT help")

# ─────────── PART B : pairwise ensemble gain ───────────
print("\n" + "=" * 78); print("PART B — does averaging a pair beat the better of the two?"); print("=" * 78)
solo = {n: roc_auc_score(y, L[n]) for n in names}
gains = []
for a, b in itertools.combinations(names, 2):
    e = (rankdata(L[a]) + rankdata(L[b])) / (2 * len(L))
    gains.append((roc_auc_score(y, e) - max(solo[a], solo[b]), a, b))
gains.sort(reverse=True)
print("  best pairings:")
for g, a, b in gains[:6]:
    print("    %+.4f   %s + %s" % (g, a[:22], b[:22]))
print("  worst pairings:")
for g, a, b in gains[-3:]:
    print("    %+.4f   %s + %s" % (g, a[:22], b[:22]))
print("\n  positive = the pair beats both members; negative = averaging hurts")

# ─────────── PART C : irreducible errors ───────────
print("\n" + "=" * 78); print("PART C — lesions EVERY model gets wrong"); print("=" * 78)
GRID = np.linspace(0.05, 0.95, 181)
wrong = np.ones(len(L), dtype=bool)
for n in names:
    t = GRID[int(np.argmax([accuracy_score(y, (L[n] > g).astype(int)) for g in GRID]))]
    wrong &= ((L[n] > t).astype(int) != y)
print("  %d of %d lesions (%.1f%%) are misclassified by ALL %d models"
      % (wrong.sum(), len(L), 100 * wrong.mean(), len(names)))
sub = L[wrong]
print("\n  their BI-RADS distribution:")
for b in sorted(sub.assessment.dropna().unique()):
    q = sub[sub.assessment == b]
    tot = int((L.assessment == b).sum())
    print("    BI-RADS %d  %3d of %-4d  (%.0f%% of that category)  malignant %.0f%%"
          % (int(b), len(q), tot, 100 * len(q) / max(tot, 1), 100 * q.label.mean()))

# ── PART D (fixed) — oracle ceiling. Run right after Cell 14. ──
import numpy as np
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score

Rk  = np.column_stack([rankdata(L[n]) / len(L) for n in names])
orc = np.where(y == 1, Rk.max(1), Rk.min(1))      # per lesion, the best any model could do
allavg = roc_auc_score(y, Rk.mean(1))
solo   = {n: roc_auc_score(y, L[n]) for n in names}
oracle = roc_auc_score(y, orc)

print("=" * 78); print("PART D — oracle: best achievable from these models"); print("=" * 78)
print("  best single model      %.4f" % max(solo.values()))
print("  average of ALL models  %.4f" % allavg)
print("  your nested ensemble   0.8991   (0.9022 before EfficientNet)")
print("  ORACLE upper bound     %.4f   <-- ceiling if selection were perfect" % oracle)
head = oracle - 0.9022
print("\n  remaining headroom from ensembling alone: %+.4f" % head)
print()
if head < 0.03:
    print("  VERDICT: little headroom. Extra backbones give marginal returns —")
    print("           stop at 0.9022 and move to writing.")
else:
    print("  VERDICT: real headroom exists -> ConvNeXt / ResNet50 worth training.")

models: 8 | lesions: 1005

PART A — Spearman correlation between models
                         endtoen phase3_ endtoen endtoen endtoen imageon      v2 efficie
endtoend                    1.00    0.67    0.73    0.69    0.55    0.57    0.69    0.65
phase3_mass                 0.67    1.00    0.92    0.93    0.74    0.72    0.87    0.85
endtoend_variants_mass:e    0.73    0.92    1.00    0.90    0.77    0.74    0.87    0.84
endtoend_variants_mass:e    0.69    0.93    0.90    1.00    0.73    0.68    0.86    0.83
endtoend_variants_mass:e    0.55    0.74    0.77    0.73    1.00    0.56    0.72    0.69
imageonly                   0.57    0.72    0.74    0.68    0.56    1.00    0.71    0.71
v2                          0.69    0.87    0.87    0.86    0.72    0.71    1.00    0.87
efficientnet_b0             0.65    0.85    0.84    0.83    0.69    0.71    0.87    1.00

  mean off-diagonal correlation: 0.75
  LOW (<0.6) means diverse models -> ensembling should help
  HIGH (>0.8) means they agr

In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 22 — BI-RADS DIAGNOSTIC  (reads mass_lesion_errors.csv, ~10 sec)
#   Decides whether BI-RADS 4 fails at RANKING or at the THRESHOLD.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D = "/root/autodl-tmp/CBIS"
L = pd.read_csv(os.path.join(D, "mass_lesion_errors.csv"))
print("loaded %d lesions | columns: %s" % (len(L), list(L.columns)))

y, p, pred = L["y"].values, L["p"].values, L["pred"].values
TOTERR = int((pred != y).sum())
print("overall: acc %.1f%%   AUC %.4f   errors %d  (FP %d, FN %d)"
      % (100 * accuracy_score(y, pred), roc_auc_score(y, p), TOTERR,
         int(((pred == 1) & (y == 0)).sum()), int(((pred == 0) & (y == 1)).sum())))

def table(col, label):
    rows = []
    for g, s in L.groupby(col):
        if len(s) < 10:
            continue
        tn, fp, fn, tp = confusion_matrix(s["y"], s["pred"], labels=[0, 1]).ravel()
        rows.append(dict(group=g, n=len(s), malig="%.0f%%" % (100 * s["y"].mean()),
                         AUC=(round(roc_auc_score(s["y"], s["p"]), 3)
                              if s["y"].nunique() > 1 else np.nan),
                         acc="%.1f%%" % (100 * accuracy_score(s["y"], s["pred"])),
                         FP=fp, FN=fn, errors=fp + fn,
                         share="%.0f%%" % (100 * (fp + fn) / max(TOTERR, 1))))
    if rows:
        print("\n--- %s ---" % label)
        print(pd.DataFrame(rows).sort_values("errors", ascending=False).to_string(index=False))

table("ass",     "BI-RADS assessment          <<< THE ONE I NEED")
table("subt_b",  "subtlety (1=subtle .. 5=obvious)")
table("dens_b",  "breast density (1=fatty .. 4=dense)")
table("marg_b",  "mass margin")
table("shape_b", "mass shape")
table("dice_q",  "segmentation quality quartile")
table("path",    "pathology label")

# ══════════════════════════════════════════════════════════════════
# BI-RADS 4 deep dive — ranking problem or threshold problem?
# ══════════════════════════════════════════════════════════════════
B = L[L["ass"] == 4].copy()
print("\n" + "=" * 78)
print("BI-RADS 4 DEEP DIVE   n=%d  (%.0f%% of cohort, %.0f%% of all errors)"
      % (len(B), 100 * len(B) / len(L), 100 * (B["pred"] != B["y"]).sum() / max(TOTERR, 1)))
print("=" * 78)
yb, pb = B["y"].values, B["p"].values
auc4 = roc_auc_score(yb, pb)
print("  malignancy rate inside BI-RADS 4        %.1f%%" % (100 * yb.mean()))
print("  AUC INSIDE BI-RADS 4                    %.4f    <<< the deciding number" % auc4)
print("  current accuracy                        %.1f%%" % (100 * accuracy_score(yb, B["pred"])))
print("  majority-class baseline                 %.1f%%" % (100 * max(yb.mean(), 1 - yb.mean())))

grid = np.linspace(0.02, 0.98, 193)
accs = np.array([accuracy_score(yb, (pb > t).astype(int)) for t in grid])
bi = int(np.argmax(accs))
print("\n  BEST POSSIBLE threshold for BI-RADS 4   %.1f%%  at t=%.3f"
      % (100 * accs[bi], grid[bi]))
print("     ^ in-sample ceiling, NOT a result — it only tells us how much of the")
print("       BI-RADS-4 error is a threshold problem vs a ranking problem.")
gain_thr = accs[bi] - accuracy_score(yb, B["pred"])
print("  recoverable by thresholding alone       %+.1f pts inside BI-RADS 4  "
      "(= %+.1f pts on the full cohort)" % (100 * gain_thr, 100 * gain_thr * len(B) / len(L)))
print("  recoverable by BETTER RANKING           %+.1f pts inside BI-RADS 4  "
      "(= %+.1f pts on the full cohort)"
      % (100 * (1 - accs[bi]), 100 * (1 - accs[bi]) * len(B) / len(L)))

print("\n  --- score distribution inside BI-RADS 4 ---")
for nm, m in [("cancers", yb == 1), ("benigns", yb == 0)]:
    q = np.percentile(pb[m], [10, 25, 50, 75, 90])
    print("     %-8s n=%3d   p10 %.3f  q1 %.3f  median %.3f  q3 %.3f  p90 %.3f"
          % (nm, m.sum(), *q))
band = ((pb > 0.40) & (pb < 0.60))
print("     lesions stuck in the 0.40-0.60 band: %d (%.0f%%), of which %d are errors"
      % (band.sum(), 100 * band.mean(), int((B["pred"].values[band] != yb[band]).sum())))

print("\n  --- is any BI-RADS-4 subgroup separable? (what a specialist could lean on) ---")
rows = []
for col, lab in [("marg_b", "margin"), ("shape_b", "shape"),
                 ("subt_b", "subtlety"), ("dens_b", "density"), ("nview", "n views")]:
    for g, s in B.groupby(col):
        if len(s) < 25 or s["y"].nunique() < 2:
            continue
        rows.append(dict(feature=lab, group=g, n=len(s),
                         malig="%.0f%%" % (100 * s["y"].mean()),
                         AUC=round(roc_auc_score(s["y"], s["p"]), 3),
                         acc="%.1f%%" % (100 * accuracy_score(s["y"], s["pred"])),
                         errors=int((s["pred"] != s["y"]).sum())))
print(pd.DataFrame(rows).sort_values("AUC", ascending=False).to_string(index=False))

print("\n  --- how much training data would a BI-RADS-4 specialist have? ---")
print("     lesions %d | images ~%d | patients %d | malignant %.0f%%"
      % (len(B), int(B["nview"].sum()), B["pid"].nunique(), 100 * yb.mean()))
print("     per fold: ~%d train / ~%d test lesions" % (int(0.8 * len(B)), int(0.2 * len(B))))

loaded 1005 lesions | columns: ['lesion_key', 'y', 'fold', 'assess', 'dens', 'subt', 'mshape', 'marg', 'path', 'dice', 'pid', 'nview', 'p', 'ass', 'pred', 'pred_global', 'wrong', 'err', 'subt_b', 'dens_b', 'dice_q', 'shape_b', 'marg_b', 'cov', 'size_q']
overall: acc 84.0%   AUC 0.8724   errors 161  (FP 96, FN 65)

--- BI-RADS assessment          <<< THE ONE I NEED ---
 group   n malig   AUC   acc  FP  FN  errors share
     4 422   48% 0.778 71.3%  68  53     121   75%
     3 213   12% 0.866 90.1%  12   9      21   13%
     0  86   14% 0.912 91.9%   4   3       7    4%
     2  59    2% 0.914 89.8%   6   0       6    4%
     5 223   97% 0.608 97.3%   6   0       6    4%

--- subtlety (1=subtle .. 5=obvious) ---
 group   n malig   AUC   acc  FP  FN  errors share
     5 396   54% 0.879 86.1%  35  20      55   34%
     4 277   38% 0.893 85.2%  24  17      41   25%
     3 210   35% 0.836 81.4%  23  16      39   24%
     2  84   56% 0.836 78.6%   8  10      18   11%
     1  37   62% 0.817 78.

In [4]:
# ══════════════════════════════════════════════════════════════════════
# CELL 28 — OPERATING-POINT CURVE  (policy C at every sensitivity floor)
#   Pick the point you will defend. Nested, full cohort, no exclusions.
#   Reads mass_lesion_errors_real.csv. CPU only, ~2 min.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D = "/root/autodl-tmp/CBIS"
FIG = os.path.join(D, "figures", "mass_final"); os.makedirs(FIG, exist_ok=True)
GRID = np.linspace(0.02, 0.98, 193)

L = pd.read_csv(os.path.join(D, "mass_lesion_errors_real.csv"))
y, p, fold, ass = L["y"].values, L["p"].values, L["fold"].values, L["ass"].values
print("n=%d   AUC %.4f   malignant %.1f%%" % (len(L), roc_auc_score(y, p), 100*y.mean()))

def sens(yy, pr): return ((pr==1)&(yy==1)).sum()/max((yy==1).sum(),1)
def thr_global(yy, pp, fl):
    a = np.array([accuracy_score(yy,(pp>t).astype(int)) for t in GRID])
    s = np.array([sens(yy,(pp>t).astype(int)) for t in GRID])
    m = s >= fl
    return float(GRID[int(np.argmax(np.where(m,a,-1.0)))]) if m.any() else float(GRID[0])
def apply_map(pp, gg, tm, dflt):
    pr = np.zeros(len(pp), int)
    for g in np.unique(gg):
        m = gg==g; pr[m] = (pp[m] > tm.get(g, dflt)).astype(int)
    return pr
def fit_joint(yy, pp, gg, fl):
    tg = thr_global(yy, pp, fl); tm = {g: tg for g in np.unique(gg)}
    def sc(t_):
        pr = apply_map(pp, gg, t_, tg)
        return accuracy_score(yy, pr) if sens(yy, pr) >= fl else -1.0
    cur = sc(tm)
    for _ in range(4):
        moved = False
        for g in np.unique(gg):
            if (gg==g).sum() < 8: continue
            bv, bs = tm[g], cur
            for v in GRID:
                t2 = dict(tm); t2[g] = v
                s = sc(t2)
                if s > bs + 1e-9: bv, bs = v, s
            if bv != tm[g]: tm[g], cur, moved = bv, bs, True
        if not moved: break
    return tm, tg

def run(fl):
    pr = np.zeros(len(L), int)
    for k in range(5):
        inn, o = (fold!=k), (fold==k)
        if o.sum()==0 or len(set(y[inn]))<2: continue
        tm, tg = fit_joint(y[inn], p[inn], ass[inn], fl)
        pr[o] = apply_map(p[o], ass[o], tm, tg)
    return pr

FLOORS = [0.55,0.60,0.65,0.70,0.75,0.80,0.85,0.90]
rows, keep = [], {}
for fl in FLOORS:
    pr = run(fl); keep[fl] = pr
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
    rows.append(dict(floor=fl, accuracy=round(100*accuracy_score(y,pr),1),
                     sensitivity=round(tp/max(tp+fn,1),3),
                     specificity=round(tn/max(tn+fp,1),3),
                     FP=fp, missed_cancers=fn))
T = pd.DataFrame(rows)
print("\n" + "="*84); print("OPERATING-POINT CURVE — nested, n=%d, nothing excluded" % len(L))
print("="*84); print(T.to_string(index=False))
print("\n  every row is defensible. pick by how many missed cancers you can justify.")

fig, ax = plt.subplots(1, 2, figsize=(13, 4.6))
ax[0].plot(T.sensitivity, T.accuracy, "o-", lw=2)
for _, r in T.iterrows():
    ax[0].annotate("%.2f" % r.floor, (r.sensitivity, r.accuracy),
                   textcoords="offset points", xytext=(5,5), fontsize=8)
ax[0].set_xlabel("sensitivity (cancers caught)"); ax[0].set_ylabel("accuracy (%)")
ax[0].set_title("accuracy vs sensitivity trade-off"); ax[0].grid(alpha=.3)
ax[1].plot(T.floor, T.missed_cancers, "o-", lw=2, color="crimson", label="missed cancers")
ax[1].plot(T.floor, T.FP, "s-", lw=2, color="steelblue", label="false alarms")
ax[1].set_xlabel("sensitivity floor"); ax[1].set_ylabel("count")
ax[1].set_title("errors by type"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout()
fp_ = os.path.join(FIG, "operating_points.png")
plt.savefig(fp_, dpi=140, bbox_inches="tight"); plt.close()
print("  figure: %s" % fp_)

print("\n" + "="*84); print("PER-BI-RADS BREAKDOWN AT EACH CANDIDATE POINT"); print("="*84)
for fl in [0.60, 0.75, 0.85]:
    pr = keep[fl]
    tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
    print("\n  floor %.2f  ->  acc %.1f%%  sens %.3f  FP %d  missed %d"
          % (fl, 100*accuracy_score(y,pr), tp/max(tp+fn,1), fp, fn))
    rr = []
    for g in np.unique(ass):
        m = ass==g
        if m.sum() < 10: continue
        a_,b_,c_,d_ = confusion_matrix(y[m], pr[m], labels=[0,1]).ravel()
        rr.append(dict(BIRADS=g, n=int(m.sum()),
                       acc="%.1f%%" % (100*accuracy_score(y[m], pr[m])),
                       sens=round(d_/max(d_+c_,1),3), FP=b_, missed=c_))
    print(pd.DataFrame(rr).to_string(index=False))

T.to_csv(os.path.join(D, "mass_operating_curve.csv"), index=False)
print("\nsaved mass_operating_curve.csv")

n=1005   AUC 0.9077   malignant 46.0%

OPERATING-POINT CURVE — nested, n=1005, nothing excluded
 floor  accuracy  sensitivity  specificity  FP  missed_cancers
  0.55      85.7        0.840        0.871  70              74
  0.60      85.7        0.840        0.871  70              74
  0.65      85.7        0.840        0.871  70              74
  0.70      85.7        0.840        0.871  70              74
  0.75      85.7        0.840        0.871  70              74
  0.80      85.7        0.840        0.871  70              74
  0.85      85.6        0.859        0.853  80              65
  0.90      85.0        0.894        0.812 102              49

  every row is defensible. pick by how many missed cancers you can justify.
  figure: /root/autodl-tmp/CBIS/figures/mass_final/operating_points.png

PER-BI-RADS BREAKDOWN AT EACH CANDIDATE POINT

  floor 0.60  ->  acc 85.7%  sens 0.840  FP 70  missed 74
 BIRADS   n   acc  sens  FP  missed
      0  86 89.5% 0.333   1       8
      2  5

In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 32 — BOOTSTRAP-STABILISED THRESHOLDS
#   The thresholds are re-fitted on 200 patient-level resamples of the
#   inner folds and the median is used. Removes threshold-picking luck.
#   Still fully nested. CPU only, ~2 min.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D = "/root/autodl-tmp/CBIS"
B = 200                                   # bootstrap resamples
GRID = np.round(np.arange(0.02, 0.981, 0.01), 3)
T = len(GRID)

L = pd.read_csv(os.path.join(D, "mass_lesion_errors_real.csv"))
y    = L["y"].values.astype(int)
p    = L["p"].values
fold = L["fold"].values.astype(int)
ass  = L["ass"].values.astype(int)
pid  = L["pid"].values
GROUPS = np.sort(np.unique(ass))
print("n=%d  AUC %.4f  malignant %.1f%%" % (len(L), roc_auc_score(y, p), 100*y.mean()))

def curves(yy, pp, gg):
    """per-group TP/TN counts at every threshold, vectorised"""
    out = {}
    for g in np.unique(gg):
        m = gg == g
        ys, ps = yy[m], pp[m]
        pr = ps[None, :] > GRID[:, None]
        out[g] = (np.asarray((pr & (ys == 1)).sum(1)),      # tp
                  np.asarray((~pr & (ys == 0)).sum(1)))     # tn
    return out

def fit(cur, P, floor, passes=8):
    """coordinate ascent over per-group thresholds under ONE sensitivity floor"""
    gs = list(cur.keys())
    tp_all = np.sum([cur[g][0] for g in gs], 0)
    cr_all = np.sum([cur[g][0] + cur[g][1] for g in gs], 0)
    ok = (tp_all / max(P, 1)) >= floor
    start = int(np.argmax(np.where(ok, cr_all, -1))) if ok.any() else int(np.argmax(cr_all))
    idx = {g: start for g in gs}
    tot_tp = int(sum(cur[g][0][idx[g]] for g in gs))
    tot_cr = int(sum(cur[g][0][idx[g]] + cur[g][1][idx[g]] for g in gs))
    for _ in range(passes):
        moved = False
        for g in gs:
            tp, tn = cur[g]
            b_tp = tot_tp - tp[idx[g]]
            b_cr = tot_cr - (tp[idx[g]] + tn[idx[g]])
            n_tp = b_tp + tp
            n_cr = b_cr + tp + tn
            feas = (n_tp / max(P, 1)) >= floor
            if not feas.any(): continue
            j = int(np.argmax(np.where(feas, n_cr, -1)))
            if j != idx[g]:
                idx[g] = j; tot_tp = int(n_tp[j]); tot_cr = int(n_cr[j]); moved = True
        if not moved: break
    return idx

def apply_idx(pp, gg, idx, dflt):
    pr = np.zeros(len(pp), int)
    for g in np.unique(gg):
        m = gg == g
        pr[m] = (pp[m] > GRID[idx.get(g, dflt)]).astype(int)
    return pr

def run(floor, bagged):
    pred = np.zeros(len(L), int)
    chosen = {}
    for k in range(5):
        inn, out = fold != k, fold == k
        if out.sum() == 0 or len(set(y[inn])) < 2: continue
        yi, pi, gi = y[inn], p[inn], ass[inn]
        pi_pid = pid[inn]
        base = fit(curves(yi, pi, gi), int((yi == 1).sum()), floor)
        if not bagged:
            idx = base
        else:
            pats = np.unique(pi_pid)
            acc = {g: [] for g in np.unique(gi)}
            rng = np.random.default_rng(0)
            for _ in range(B):
                take = rng.choice(pats, size=len(pats), replace=True)
                sel = np.concatenate([np.where(pi_pid == q)[0] for q in take])
                yb, pb, gb = yi[sel], pi[sel], gi[sel]
                if len(set(yb)) < 2: continue
                fb = fit(curves(yb, pb, gb), int((yb == 1).sum()), floor)
                for g in fb: acc[g].append(fb[g])
            idx = {g: (int(np.median(v)) if len(v) >= B // 4 else base.get(g, len(GRID)//2))
                   for g, v in acc.items()}
        dflt = int(np.median(list(idx.values())))
        pred[out] = apply_idx(p[out], ass[out], idx, dflt)
        chosen[k] = {int(g): float(GRID[i]) for g, i in sorted(idx.items())}
    return pred, chosen

print("\n" + "="*86)
print("SINGLE FIT  vs  BOOTSTRAP-STABILISED  (nested, full cohort, nothing excluded)")
print("="*86)
rows, keep = [], {}
for floor in [0.60, 0.75, 0.85]:
    for bag in [False, True]:
        pr, ch = run(floor, bag)
        tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
        rows.append(dict(floor=floor, thresholds=("bootstrap x%d" % B) if bag else "single fit",
                         accuracy="%.1f%%" % (100*accuracy_score(y, pr)),
                         sens=round(tp/max(tp+fn,1),3), spec=round(tn/max(tn+fp,1),3),
                         FP=fp, missed=fn))
        keep[(floor, bag)] = (pr, ch)
print(pd.DataFrame(rows).to_string(index=False))

print("\n--- how much do the chosen thresholds move between folds? ---")
for bag in [False, True]:
    _, ch = keep[(0.60, bag)]
    gs = sorted({g for f in ch.values() for g in f})
    sd = {g: float(np.std([ch[k][g] for k in ch if g in ch[k]])) for g in gs}
    print("  %-16s  per-group spread across folds: %s"
          % (("bootstrap" if bag else "single fit"),
             {g: round(v, 3) for g, v in sd.items()}))
    print("                    mean spread %.4f  <-- smaller is more stable"
          % np.mean(list(sd.values())))

BEST = (0.60, True)
pr, ch = keep[BEST]
L["pred"] = pr
tn, fp, fn, tp = confusion_matrix(y, pr, labels=[0,1]).ravel()
print("\n" + "="*86)
print("BOOTSTRAP-STABILISED, floor %.2f   acc %.1f%%  sens %.3f  spec %.3f  FP %d  missed %d"
      % (BEST[0], 100*accuracy_score(y, pr), tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))
print("="*86)
TE = int((pr != y).sum()); out = []
for g in GROUPS:
    m = ass == g
    if m.sum() < 10: continue
    a_, b_, c_, d_ = confusion_matrix(y[m], pr[m], labels=[0,1]).ravel()
    out.append(dict(BIRADS=g, n=int(m.sum()),
                    malignant=int((y[m]==1).sum()), benign=int((y[m]==0).sum()),
                    AUC=round(roc_auc_score(y[m], p[m]), 3) if len(set(y[m]))>1 else np.nan,
                    accuracy="%.1f%%" % (100*accuracy_score(y[m], pr[m])),
                    sens=round(d_/max(d_+c_,1),3), FP=b_, missed=c_,
                    share="%.0f%%" % (100*(b_+c_)/max(TE,1))))
print(pd.DataFrame(out).to_string(index=False))
print("\nNote: AUC is uninformative where one class is tiny (BI-RADS 5 has %d benign, "
      "BI-RADS 2 has %d malignant)." % (int(((ass==5)&(y==0)).sum()), int(((ass==2)&(y==1)).sum())))
L.to_csv(os.path.join(D, "mass_lesion_errors_final.csv"), index=False)
print("saved mass_lesion_errors_final.csv")

n=1005  AUC 0.9077  malignant 46.0%

SINGLE FIT  vs  BOOTSTRAP-STABILISED  (nested, full cohort, nothing excluded)
 floor     thresholds accuracy  sens  spec  FP  missed
  0.60     single fit    85.7% 0.842 0.869  71      73
  0.60 bootstrap x200    85.6% 0.844 0.866  73      72
  0.75     single fit    85.7% 0.842 0.869  71      73
  0.75 bootstrap x200    85.6% 0.844 0.866  73      72
  0.85     single fit    86.0% 0.868 0.853  80      61
  0.85 bootstrap x200    85.6% 0.866 0.847  83      62

--- how much do the chosen thresholds move between folds? ---
  single fit        per-group spread across folds: {0: 0.012, 1: 0.0, 2: 0.004, 3: 0.008, 4: 0.047, 5: 0.0}
                    mean spread 0.0119  <-- smaller is more stable
  bootstrap         per-group spread across folds: {0: 0.015, 1: 0.0, 2: 0.004, 3: 0.008, 4: 0.035, 5: 0.0}
                    mean spread 0.0103  <-- smaller is more stable

BOOTSTRAP-STABILISED, floor 0.60   acc 85.6%  sens 0.844  spec 0.866  FP 73  missed 72

In [1]:
# ══════════════════════════════════════════════════════════════════════
# CELL 33 — EVALUATE ON THE OFFICIAL CBIS-DDSM TRAIN/TEST SPLIT
#   Thresholds are fitted ONLY on official-train lesions and applied
#   unchanged to official-test lesions. CPU only, ~1 min.
# ══════════════════════════════════════════════════════════════════════
import os
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
pd.set_option("display.width", 220)

D, LES = "/root/autodl-tmp/CBIS", "mass"
GRID = np.round(np.arange(0.02, 0.981, 0.01), 3)

d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES))
d["lesion_key"] = d["lesion_key"].astype(str)
assert "official_split" in d.columns, "no official_split column"
d["osp"] = d["official_split"].astype(str).str.lower().str.strip()
print("official_split values:", d["osp"].value_counts().to_dict())

sp = d.groupby("lesion_key")["osp"].agg(lambda s: s.mode().iat[0])
L = pd.read_csv(os.path.join(D, "mass_lesion_errors_final.csv"))
L["lesion_key"] = L["lesion_key"].astype(str)
L["osp"] = L["lesion_key"].map(sp)
L = L.dropna(subset=["osp"]).reset_index(drop=True)

istr = L["osp"].str.contains("train")
iste = L["osp"].str.contains("test")
print("\nlesions: official-train %d | official-test %d | unmapped %d"
      % (istr.sum(), iste.sum(), len(L) - istr.sum() - iste.sum()))
ov = set(L.loc[istr, "pid"]) & set(L.loc[iste, "pid"])
print("patients in BOTH official train and test: %d  %s" % (len(ov), "(must be 0)" if not ov else "<-- PROBLEM"))

y, p, ass = L["y"].values, L["p"].values, L["ass"].values

def sens(yy, pr): return ((pr==1)&(yy==1)).sum()/max((yy==1).sum(),1)
def curves(yy, pp, gg):
    out = {}
    for g in np.unique(gg):
        m = gg==g; ys, ps = yy[m], pp[m]
        pr = ps[None,:] > GRID[:,None]
        out[g] = (np.asarray((pr & (ys==1)).sum(1)), np.asarray((~pr & (ys==0)).sum(1)))
    return out
def fit(cur, P, floor, passes=8):
    gs = list(cur.keys())
    tp_a = np.sum([cur[g][0] for g in gs], 0); cr_a = np.sum([cur[g][0]+cur[g][1] for g in gs], 0)
    ok = (tp_a/max(P,1)) >= floor
    s0 = int(np.argmax(np.where(ok, cr_a, -1))) if ok.any() else int(np.argmax(cr_a))
    idx = {g: s0 for g in gs}
    ttp = int(sum(cur[g][0][idx[g]] for g in gs))
    tcr = int(sum(cur[g][0][idx[g]]+cur[g][1][idx[g]] for g in gs))
    for _ in range(passes):
        moved = False
        for g in gs:
            tp, tn = cur[g]
            n_tp = ttp - tp[idx[g]] + tp
            n_cr = tcr - (tp[idx[g]]+tn[idx[g]]) + tp + tn
            fe = (n_tp/max(P,1)) >= floor
            if not fe.any(): continue
            j = int(np.argmax(np.where(fe, n_cr, -1)))
            if j != idx[g]: idx[g]=j; ttp=int(n_tp[j]); tcr=int(n_cr[j]); moved=True
        if not moved: break
    return idx

FLOOR = 0.60
idx = fit(curves(y[istr.values], p[istr.values], ass[istr.values]),
          int((y[istr.values]==1).sum()), FLOOR)
dflt = int(np.median(list(idx.values())))
print("\nthresholds fitted on official-train:",
      {int(g): float(GRID[i]) for g, i in sorted(idx.items())})

def report(mask, tag):
    yy, pp, gg = y[mask], p[mask], ass[mask]
    pr = np.zeros(len(yy), int)
    for g in np.unique(gg):
        m = gg==g; pr[m] = (pp[m] > GRID[idx.get(g, dflt)]).astype(int)
    tn, fp, fn, tp = confusion_matrix(yy, pr, labels=[0,1]).ravel()
    print("  %-26s n=%4d  malig %.0f%%  AUC %.4f  acc %.1f%%  sens %.3f  spec %.3f  FP %3d  FN %3d"
          % (tag, len(yy), 100*yy.mean(), roc_auc_score(yy, pp), 100*accuracy_score(yy, pr),
             tp/max(tp+fn,1), tn/max(tn+fp,1), fp, fn))
    return pr

print("\n" + "="*104)
print("OFFICIAL CBIS-DDSM SPLIT")
print("="*104)
report(istr.values, "official TRAIN (in-sample)")
prt = report(iste.values, "official TEST (held out)")
report(np.ones(len(L), bool), "all lesions (5-fold CV)")

print("\n--- official TEST, by BI-RADS ---")
yt, at = y[iste.values], ass[iste.values]
rows = []
for g in np.unique(at):
    m = at==g
    if m.sum() < 5: continue
    a_,b_,c_,d_ = confusion_matrix(yt[m], prt[m], labels=[0,1]).ravel()
    rows.append(dict(BIRADS=int(g), n=int(m.sum()), malig=int((yt[m]==1).sum()),
                     AUC=round(roc_auc_score(yt[m], p[iste.values][m]),3) if len(set(yt[m]))>1 else np.nan,
                     acc="%.1f%%" % (100*accuracy_score(yt[m], prt[m])), FP=b_, missed=c_))
print(pd.DataFrame(rows).to_string(index=False))

print("\nIMPORTANT — how to describe this in the thesis:")
print("  These are out-of-fold predictions from patient-grouped 5-fold CV, RESTRICTED to the")
print("  official test lesions, with thresholds fitted only on the official training lesions.")
print("  Every lesion is still scored by a model that never saw its patient, so the number is")
print("  leakage-free. It is NOT a reproduction of the official protocol (which would require")
print("  retraining on the official training set alone) — call it 'official test subset under")
print("  our CV protocol', not 'official split result'.")

official_split values: {'train': 1318, 'test': 378}

lesions: official-train 782 | official-test 223 | unmapped 0
patients in BOTH official train and test: 0  (must be 0)

thresholds fitted on official-train: {0: 0.54, 1: 0.02, 2: 0.8, 3: 0.59, 4: 0.48, 5: 0.02}

OFFICIAL CBIS-DDSM SPLIT
  official TRAIN (in-sample) n= 782  malig 48%  AUC 0.9150  acc 87.6%  sens 0.888  spec 0.865  FP  55  FN  42
  official TEST (held out)   n= 223  malig 39%  AUC 0.8791  acc 81.6%  sens 0.816  spec 0.816  FP  25  FN  16
  all lesions (5-fold CV)    n=1005  malig 46%  AUC 0.9077  acc 86.3%  sens 0.874  spec 0.853  FP  80  FN  58

--- official TEST, by BI-RADS ---
 BIRADS  n  malig   AUC   acc  FP  missed
      0 18      2 0.906 77.8%   3       1
      2 10      1 1.000 90.0%   0       1
      3 53      2 0.980 92.5%   4       0
      4 96     38 0.750 68.8%  16      14
      5 45     43 0.640 95.6%   2       0

IMPORTANT — how to describe this in the thesis:
  These are out-of-fold predictions from pati

In [5]:
# ══════════════════════════════════════════════════════════════════════
# CELL 34 — CALCIFICATION PRETRAINING -> TWO-STREAM FINE-TUNE
#   Stage 1: DenseNet-121 trained on 1,635 calcification lesions whose
#            patients appear NOWHERE in the mass dataset (leak-free for
#            all folds). Backbone saved to disk.
#   Stage 2: both two-stream backbones initialised from it, fine-tuned
#            and evaluated on mass only. Zero calc lesions in any test set.
#   >>> QUICK_TEST = True FIRST <<<   full run ~5 h
# ══════════════════════════════════════════════════════════════════════
import os, gc, time
os.environ.setdefault("OMP_NUM_THREADS", "4")
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D = "/root/autodl-tmp/CBIS"
DEV = torch.device("cuda")
torch.backends.cudnn.benchmark = True; torch.backends.cuda.matmul.allow_tf32 = True

QUICK_TEST = False                     # <<<<<< set False for the real run

CKPT   = os.path.join(D, "calc_pretrained_backbone.pt")
OUTCSV = os.path.join(D, "cv_mass_twostream_calcpre_oof.csv")
ST, SW, BATCH = 512, 384, 8
SEED = 11
PRE_EPOCHS, PRE_BATCH, PRE_LR = 14, 12, 3e-4
EPOCHS, FREEZE = 20, 3
LR_HEAD, LR_HEAD_FT, LR_BACK = 1e-3, 3e-4, 3e-5
WD, GAMMA, AUX_W, MULT, PATIENCE, ATT = 1e-4, 2.0, 0.3, 3, 7, 2.0
FOLDS = [0, 1, 2, 3, 4]
if QUICK_TEST:
    PRE_EPOCHS, EPOCHS, FREEZE, MULT, FOLDS = 3, 4, 1, 1, [0]
    print(">>> QUICK TEST: 3 pretrain epochs, 1 fold, 4 epochs — error check only\n")

MEAN = np.array([0.485,0.456,0.406], np.float32).reshape(3,1,1)
STD  = np.array([0.229,0.224,0.225], np.float32).reshape(3,1,1)
cl = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
def load(p_, size, mask=False):
    im = cv2.imread(str(p_), cv2.IMREAD_GRAYSCALE)
    if im is None: im = np.zeros((size,size), np.uint8)
    if im.shape != (size,size):
        im = cv2.resize(im, (size,size),
                        interpolation=cv2.INTER_NEAREST if mask else cv2.INTER_AREA)
    return (im>127).astype(np.uint8) if mask else cl.apply(im)

def pool2(feat, m, att):
    mm = F.interpolate(m, size=feat.shape[2:], mode="bilinear", align_corners=False)
    w = 1.0 + att*mm
    return torch.cat([(feat*w).sum((2,3))/(w.sum((2,3))+1e-6), feat.mean((2,3))], 1)

def backbone():
    try:  return models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1).features
    except Exception: return models.densenet121(weights=None).features

def focal(lg, tg, al):
    ce = F.cross_entropy(lg.float(), tg, weight=al, reduction="none")
    return ((1-torch.exp(-ce))**GAMMA * ce).mean()

# ══════════════════════════════════════════════════════════════════
# STAGE 1 — pretrain on calcification
# ══════════════════════════════════════════════════════════════════
if os.path.exists(CKPT) and not QUICK_TEST:
    print("stage 1 skipped — checkpoint already exists (%s)" % os.path.basename(CKPT))
else:
    m_ = pd.read_csv(os.path.join(D, "unified_folds_mass.csv"))
    c  = pd.read_csv(os.path.join(D, "unified_folds_calc.csv"))
    c  = c[~c.patient_id.isin(set(m_.patient_id))].reset_index(drop=True)
    PMC = os.path.join(D, "predmasks_calc")
    c["stem"] = c["img"].apply(lambda q: os.path.basename(str(q)).replace("_img.png",""))
    c["pm"]   = c["stem"].apply(lambda s: os.path.join(PMC, s+"_pred.png"))
    c = c[c["pm"].apply(os.path.exists) & c["img"].apply(os.path.exists)].reset_index(drop=True)
    c["label"] = c["label"].astype(int)
    print("STAGE 1: %d calc images | %d patients | malignant %.1f%%"
          % (len(c), c.patient_id.nunique(), 100*c.label.mean()))
    assert len(c) > 500, "too few clean calc images"

    rng = np.random.default_rng(0)
    pats = np.array(sorted(c.patient_id.unique())); rng.shuffle(pats)
    vp = set(pats[:max(int(0.12*len(pats)), 10)])
    tr_i = np.where(~c.patient_id.isin(vp))[0]
    va_i = np.where( c.patient_id.isin(vp))[0]
    print("  pretrain split: %d train / %d val images (patient-disjoint)" % (len(tr_i), len(va_i)))

    CC, t0 = {}, time.time()
    for _, r in c.iterrows():
        CC[r["stem"]] = (load(r["img"], ST), load(r["pm"], ST, True))
    print("  cached %d calc images in %.0fs" % (len(CC), time.time()-t0))

    class CDS(Dataset):
        def __init__(self, idx, aug): self.idx, self.aug = np.asarray(idx), aug
        def __len__(self): return len(self.idx)
        def __getitem__(self, i):
            j = int(self.idx[i]); r = c.iloc[j]
            im, mk = [a.copy() for a in CC[r["stem"]]]
            if self.aug:
                if np.random.rand()<.5: im, mk = im[:,::-1], mk[:,::-1]
                if np.random.rand()<.5: im, mk = im[::-1,:], mk[::-1,:]
                kk = np.random.randint(4)
                if kk: im, mk = np.rot90(im,kk), np.rot90(mk,kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if np.random.rand()<.7:
                    M = cv2.getRotationMatrix2D((ST/2,ST/2), np.random.uniform(-25,25),
                                                np.random.uniform(.9,1.12))
                    im = cv2.warpAffine(im, M, (ST,ST), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (ST,ST), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if np.random.rand()<.5:
                    im = np.clip(im.astype(np.float32)*np.random.uniform(.85,1.15)
                                 + np.random.uniform(-12,12), 0, 255).astype(np.uint8)
            im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
            f = im.astype(np.float32)/255.0
            x = ((np.stack([f]*3,0)-MEAN)/STD).astype(np.float32)
            return (torch.from_numpy(x),
                    torch.from_numpy(mk.astype(np.float32))[None],
                    torch.tensor(int(r["label"])))

    class PreNet(nn.Module):
        def __init__(self):
            super().__init__()
            self.b = backbone()
            self.head = nn.Sequential(nn.Linear(2048,512), nn.BatchNorm1d(512), nn.ReLU(True),
                                      nn.Dropout(0.4), nn.Linear(512,2))
        def forward(self, x, m):
            return self.head(pool2(F.relu(self.b(x)), m, ATT))

    torch.manual_seed(SEED); np.random.seed(SEED)
    net = PreNet().to(DEV)
    yv = c["label"].values
    n0, n1 = float((yv[tr_i]==0).sum()), float((yv[tr_i]==1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    opt = torch.optim.AdamW(net.parameters(), lr=PRE_LR, weight_decay=WD)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=PRE_EPOCHS)
    scaler = torch.amp.GradScaler()
    tl = DataLoader(CDS(tr_i, True), batch_size=PRE_BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    vl = DataLoader(CDS(va_i, False), batch_size=16, shuffle=False, num_workers=0)

    best, bstate = -1.0, None
    for ep in range(1, PRE_EPOCHS+1):
        net.train()
        for x, m, t_ in tl:
            x, m, t_ = x.to(DEV, non_blocking=True), m.to(DEV, non_blocking=True), t_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                loss = focal(net(x, m), t_, al)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        sch.step()
        net.eval(); ps = []
        with torch.no_grad():
            for x, m, _ in vl:
                with torch.amp.autocast(device_type="cuda"):
                    o = net(x.to(DEV), m.to(DEV))
                ps += list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
        a = roc_auc_score(yv[va_i], np.array(ps)) if len(set(yv[va_i]))>1 else 0.0
        star = ""
        if a > best:
            best = a; star = " *"
            bstate = {k: v.detach().cpu().clone() for k, v in net.b.state_dict().items()}
        print("    pretrain ep %2d  calc val-AUC %.4f%s" % (ep, a, star), flush=True)

    torch.save(bstate, CKPT)
    print("\n  STAGE 1 done. best calc val-AUC %.4f  ->  %s" % (best, os.path.basename(CKPT)))
    print("  GATE: if this is below ~0.62 the backbone learned little and stage 2 will not help.")
    del net, CC; gc.collect(); torch.cuda.empty_cache()

# ══════════════════════════════════════════════════════════════════
# STAGE 2 — two-stream fine-tune on MASS, backbones pre-initialised
# ══════════════════════════════════════════════════════════════════
LES = "mass"
WIDE = os.path.join(D, "crops_wide_%s" % LES); PM = os.path.join(D, "predmasks_%s" % LES)
d = pd.read_csv(os.path.join(D, "unified_folds_%s.csv" % LES)).reset_index(drop=True)
d["label"] = d["label"].astype(int)
d["stem"]  = d["img"].apply(lambda q: os.path.basename(str(q)).replace("_img.png",""))
d["tmask"] = d["stem"].apply(lambda s: os.path.join(PM,   s+"_pred.png"))
d["wimg"]  = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_img.png"))
d["wmask"] = d["stem"].apply(lambda s: os.path.join(WIDE, s+"_pred.png"))
assert (~d["wimg"].apply(os.path.exists)).sum() == 0, "wide crops missing — run Cell 29c"
print("\nSTAGE 2: %d mass images | %d patients | malignant %.1f%%"
      % (len(d), d.patient_id.nunique(), 100*d.label.mean()))

CACHE, t0 = {}, time.time()
for _, r in d.iterrows():
    CACHE[r["stem"]] = (load(r["img"], ST), load(r["tmask"], ST, True),
                        load(r["wimg"], SW), load(r["wmask"], SW, True))
print("  cached %d in %.0fs" % (len(CACHE), time.time()-t0))

def primary(x): return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()
aux, meta = {}, {}
for cc in ["subtlety","mass_shape","mass_margins"]:
    if cc not in d.columns or d[cc].notna().sum()==0: continue
    if cc == "subtlety":
        v = pd.to_numeric(d[cc], errors="coerce").where(lambda z:(z>=1)&(z<=5))
        codes, n = (v-1).fillna(-1).astype(int).values, 5
    else:
        pr = d[cc].map(primary)
        pr = pr.where(pr.isin(pr.value_counts().head(6).index.tolist()), "OTHER")
        cats = sorted([q for q in pr.unique() if q != "UNK"]); mp = {q:i for i,q in enumerate(cats)}
        codes, n = pr.map(lambda z: mp.get(z,-1)).astype(int).values, len(cats)
    if n > 1: aux[cc], meta[cc] = codes, n
AK = sorted(aux.keys()); print("  helper heads:", meta)

class DS(Dataset):
    def __init__(self, idx, aug, mult=1, tta=0):
        self.idx, self.aug, self.mult, self.tta = np.asarray(idx), aug, (mult if aug else 1), tta
    def __len__(self): return len(self.idx)*self.mult
    def __getitem__(self, i):
        j = int(self.idx[i % len(self.idx)]); r = d.iloc[j]
        ti, tm, wi, wm = [a.copy() for a in CACHE[r["stem"]]]
        if self.aug:
            fh, fv = np.random.rand()<.5, np.random.rand()<.5
            kk = np.random.randint(4); aff = np.random.rand()<.7
            ang, sc = np.random.uniform(-25,25), np.random.uniform(.9,1.12)
            itn = np.random.rand()<.5
            gg, bb = np.random.uniform(.85,1.15), np.random.uniform(-12,12)
            def T(im, mk, s):
                if fh: im, mk = im[:,::-1], mk[:,::-1]
                if fv: im, mk = im[::-1,:], mk[::-1,:]
                if kk: im, mk = np.rot90(im,kk), np.rot90(mk,kk)
                im, mk = np.ascontiguousarray(im), np.ascontiguousarray(mk)
                if aff:
                    M = cv2.getRotationMatrix2D((s/2,s/2), ang, sc)
                    im = cv2.warpAffine(im, M, (s,s), flags=cv2.INTER_LINEAR,
                                        borderMode=cv2.BORDER_REFLECT)
                    mk = cv2.warpAffine(mk, M, (s,s), flags=cv2.INTER_NEAREST,
                                        borderMode=cv2.BORDER_CONSTANT)
                if itn: im = np.clip(im.astype(np.float32)*gg+bb, 0, 255).astype(np.uint8)
                return im, mk
            ti, tm = T(ti, tm, ST); wi, wm = T(wi, wm, SW)
        else:
            t = self.tta
            def V(im, mk):
                if   t==1: return im[:,::-1], mk[:,::-1]
                elif t==2: return im[::-1,:], mk[::-1,:]
                elif t==3: return np.rot90(im,2), np.rot90(mk,2)
                return im, mk
            ti, tm = V(ti, tm); wi, wm = V(wi, wm)
        o = []
        for im, mk in [(ti,tm),(wi,wm)]:
            f = np.ascontiguousarray(im).astype(np.float32)/255.0
            o.append(torch.from_numpy(((np.stack([f]*3,0)-MEAN)/STD).astype(np.float32)))
            o.append(torch.from_numpy(np.ascontiguousarray(mk).astype(np.float32))[None])
        av = (np.array([aux[q][j] for q in AK], dtype=np.int64) if AK else np.zeros(0, np.int64))
        return o[0], o[1], o[2], o[3], torch.tensor(int(r["label"])), torch.from_numpy(av)

class TwoStream(nn.Module):
    def __init__(self, am, pre=None):
        super().__init__()
        self.bt, self.bw = backbone(), backbone()
        if pre is not None:
            self.bt.load_state_dict(pre); self.bw.load_state_dict(pre)
        Fd = 1024*4
        self.head = nn.Sequential(nn.Linear(Fd,512), nn.BatchNorm1d(512), nn.ReLU(True),
                                  nn.Dropout(0.4), nn.Linear(512,2))
        self.keys = sorted(am.keys())
        self.aux = nn.ModuleList([nn.Sequential(nn.Linear(Fd,128), nn.ReLU(True),
                                                nn.Dropout(0.3), nn.Linear(128, am[q]))
                                  for q in self.keys])
    def forward(self, xt, mt, xw, mw):
        g = torch.cat([pool2(F.relu(self.bt(xt)), mt, ATT),
                       pool2(F.relu(self.bw(xw)), mw, ATT)], 1)
        return self.head(g), [h(g) for h in self.aux]

PRE = torch.load(CKPT, map_location="cpu")
print("  loaded pretrained backbone (%d tensors)" % len(PRE))

@torch.no_grad()
def predict(net, idx, tta=True):
    net.eval(); tot = None
    for t in ([0,1,2,3] if tta else [0]):
        ld = DataLoader(DS(idx, False, 1, tta=t), batch_size=12, shuffle=False, num_workers=0)
        ps = []
        for xt, mt, xw, mw, _, _ in ld:
            with torch.amp.autocast(device_type="cuda"):
                o, _ = net(xt.to(DEV), mt.to(DEV), xw.to(DEV), mw.to(DEV))
            ps += list(torch.softmax(o.float(),1)[:,1].cpu().numpy())
        ps = np.array(ps); tot = ps if tot is None else tot+ps
    return tot/(4 if tta else 1)

def train_fold(tr, va, te):
    torch.manual_seed(SEED); np.random.seed(SEED)
    yy = d["label"].values
    n0, n1 = float((yy[tr]==0).sum()), float((yy[tr]==1).sum())
    al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV, dtype=torch.float32)
    net = TwoStream(meta, PRE).to(DEV)
    back = list(net.bt.parameters()) + list(net.bw.parameters())
    for q in back: q.requires_grad = False
    hp = [q for n_, q in net.named_parameters()
          if not (n_.startswith("bt.") or n_.startswith("bw."))]
    scaler = torch.amp.GradScaler()
    opt = torch.optim.AdamW(hp, lr=LR_HEAD, weight_decay=WD); sch = None
    tl = DataLoader(DS(tr, True, MULT), batch_size=BATCH, shuffle=True,
                    num_workers=0, pin_memory=True, drop_last=True)
    best, bstate, bad = -1.0, None, 0
    for ep in range(1, EPOCHS+1):
        if ep == FREEZE+1:
            for q in back: q.requires_grad = True
            opt = torch.optim.AdamW([{"params": back, "lr": LR_BACK},
                                     {"params": hp,   "lr": LR_HEAD_FT}], weight_decay=WD)
            sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max(1, EPOCHS-FREEZE))
        net.train()
        if ep <= FREEZE: net.bt.eval(); net.bw.eval()
        for xt, mt, xw, mw, t_, a_ in tl:
            xt, mt = xt.to(DEV, non_blocking=True), mt.to(DEV, non_blocking=True)
            xw, mw = xw.to(DEV, non_blocking=True), mw.to(DEV, non_blocking=True)
            t_, a_ = t_.to(DEV), a_.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type="cuda"):
                o, ax = net(xt, mt, xw, mw)
                loss = focal(o, t_, al)
                if len(ax):
                    loss = loss + AUX_W*sum(F.cross_entropy(q.float(), a_[:,h], ignore_index=-1)
                                            for h, q in enumerate(ax))/len(ax)
            if not torch.isfinite(loss): continue
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
            scaler.step(opt); scaler.update()
        if sch is not None: sch.step()
        pv = predict(net, va, tta=False)
        a = roc_auc_score(yy[va], pv) if len(set(yy[va]))>1 else 0.0
        star = ""
        if a > best:
            best, bad = a, 0; star = " *"
            bstate = {k: v.detach().cpu().clone() for k, v in net.state_dict().items()}
        else: bad += 1
        print("      ep %2d  val-AUC %.4f%s" % (ep, a, star), flush=True)
        if bad >= PATIENCE: print("      early stop"); break
    net.load_state_dict({k: v.to(DEV) for k, v in bstate.items()})
    pt = predict(net, te, tta=True)
    del net; gc.collect(); torch.cuda.empty_cache()
    return pt, best

y = d["label"].values; oof = np.full(len(d), np.nan)
for kf in FOLDS:
    role = d["role_f%d" % kf]
    tr, va, te = np.where(role=="train")[0], np.where(role=="val")[0], np.where(role=="test")[0]
    assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "LEAK"
    print("\n### fold %d | train %d val %d test %d" % (kf, len(tr), len(va), len(te)))
    t0 = time.time()
    p_, bv = train_fold(tr, va, te)
    oof[te] = p_
    print("  FOLD %d best val %.4f | test AUC %.4f  (%.0fs)"
          % (kf, bv, roc_auc_score(y[te], p_), time.time()-t0))

done = ~np.isnan(oof)
res = d.loc[done, ["img","lesion_key","label"]].copy(); res["prob"] = oof[done]
Lg = res.groupby("lesion_key").agg(y=("label","max"), p=("prob","mean")).reset_index()
print("\n" + "="*72); print("RESULT"); print("="*72)
print("  per-image  AUC %.4f  (n=%d)" % (roc_auc_score(y[done], oof[done]), done.sum()))
print("  per-lesion AUC %.4f" % roc_auc_score(Lg.y, Lg.p))
print("  COMPARE AGAINST: two-stream seed 11 WITHOUT pretraining = 0.8652")
print("  (same seed, same folds, same data — the only change is the initialisation)")
if not QUICK_TEST:
    res.rename(columns={"label":"true"}).to_csv(OUTCSV, index=False)
    print("\n  saved %s  -> rerun Cell 27" % os.path.basename(OUTCSV))
else:
    print("\n  QUICK TEST clean. Set QUICK_TEST = False and rerun (~5 h).")

STAGE 1: 1635 calc images | 674 patients | malignant 35.2%
  pretrain split: 1397 train / 238 val images (patient-disjoint)
  cached 1635 calc images in 8s
    pretrain ep  1  calc val-AUC 0.6417 *
    pretrain ep  2  calc val-AUC 0.7994 *
    pretrain ep  3  calc val-AUC 0.7573
    pretrain ep  4  calc val-AUC 0.7780
    pretrain ep  5  calc val-AUC 0.7576
    pretrain ep  6  calc val-AUC 0.8236 *
    pretrain ep  7  calc val-AUC 0.8176
    pretrain ep  8  calc val-AUC 0.7968
    pretrain ep  9  calc val-AUC 0.8312 *
    pretrain ep 10  calc val-AUC 0.8043
    pretrain ep 11  calc val-AUC 0.8481 *
    pretrain ep 12  calc val-AUC 0.8499 *
    pretrain ep 13  calc val-AUC 0.8498
    pretrain ep 14  calc val-AUC 0.8467

  STAGE 1 done. best calc val-AUC 0.8499  ->  calc_pretrained_backbone.pt
  GATE: if this is below ~0.62 the backbone learned little and stage 2 will not help.

STAGE 2: 1696 mass images | 892 patients | malignant 46.2%
  cached 1696 in 17s
  helper heads: {'subtlety': 5

In [1]:
# ══ Stage 2 parameter counts (no weights downloaded, no GPU) ══
import torch, torch.nn as nn
from torchvision import models

def build(name):
    if name == "densenet121":     m = models.densenet121(weights=None);     return m.features, 1024
    if name == "convnext_tiny":   m = models.convnext_tiny(weights=None);   return m.features, 768
    if name == "efficientnet_b0": m = models.efficientnet_b0(weights=None); return m.features, 1280
    if name == "resnet50":
        m = models.resnet50(weights=None); return nn.Sequential(*list(m.children())[:-2]), 2048
    raise ValueError(name)

AUX = {"subtlety": 5, "mass_shape": 6, "mass_margins": 6}
n = lambda m: sum(p.numel() for p in m.parameters())

def head_params(feat, streams=1, dual=True):
    d = feat * streams * (2 if dual else 1)
    main = nn.Sequential(nn.Linear(d, 512), nn.BatchNorm1d(512), nn.ReLU(True),
                         nn.Dropout(0.4), nn.Linear(512, 2))
    aux = nn.ModuleList([nn.Sequential(nn.Linear(d, 128), nn.ReLU(True),
                                       nn.Dropout(0.3), nn.Linear(128, k))
                         for k in AUX.values()])
    return n(main), n(aux)

print(f"{'configuration':<46}{'backbone':>10}{'head':>8}{'INFER M':>10}{'aux(train)':>12}")
print("-" * 86)
for name in ["convnext_tiny", "densenet121", "efficientnet_b0", "resnet50"]:
    f, fd = build(name); bp = n(f)
    for st, dual, lab in [(1, False, "single-stream, global pool only (control)"),
                          (1, True,  "single-stream, DUAL pooling"),
                          (2, True,  "two-stream, DUAL pooling")]:
        hp, ap = head_params(fd, st, dual)
        print(f"{name + '  ' + lab:<46}{bp*st/1e6:>10.2f}{hp/1e6:>8.2f}"
              f"{(bp*st+hp)/1e6:>10.2f}{ap/1e6:>12.2f}")

configuration                                   backbone    head   INFER M  aux(train)
--------------------------------------------------------------------------------------
convnext_tiny  single-stream, global pool only (control)     27.82    0.40     28.21        0.30
convnext_tiny  single-stream, DUAL pooling         27.82    0.79     28.61        0.59
convnext_tiny  two-stream, DUAL pooling            55.64    1.58     57.21        1.18
densenet121  single-stream, global pool only (control)      6.95    0.53      7.48        0.40
densenet121  single-stream, DUAL pooling            6.95    1.05      8.00        0.79
densenet121  two-stream, DUAL pooling              13.91    2.10     16.01        1.58
efficientnet_b0  single-stream, global pool only (control)      4.01    0.66      4.67        0.49
efficientnet_b0  single-stream, DUAL pooling        4.01    1.31      5.32        0.99
efficientnet_b0  two-stream, DUAL pooling           8.02    2.62     10.64        1.97
resnet50  sin

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# BACKBONE ABLATION — is dual pooling backbone-agnostic?
#   Grid: {convnext_tiny, densenet121, efficientnet_b0, resnet50}
#         x {blind, dual}.  Identical folds, aug, TTA, loss, seeds.
#     blind : g = mean(f)
#     dual  : w=1+2m ; f_g=sum(w*f)/sum(w) ; f_u=mean(f) ; g=[f_g||f_u]
#   RESULT = the gain (dual - blind) per backbone.
#   QUICK=True absolute AUCs will NOT match your 0.8692 — never mix rows.
#   Self-contained. Resumable. LES forced to "mass".
# ══════════════════════════════════════════════════════════════════════
import os, time
os.environ["OMP_NUM_THREADS"] = "4"
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, pandas as pd, cv2
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from sklearn.metrics import roc_auc_score
cv2.setNumThreads(0)

D   = "/root/autodl-tmp/CBIS"
DEV = torch.device("cuda")
OUT = os.path.join(D, "backbone_ablation"); os.makedirs(OUT, exist_ok=True)

QUICK          = True     # True: S=448,MULT=4,EP=12 (~4h)  False: 512/8/20 (~12h)
USE_PRED_MASKS = True     # True = predicted masks (end-to-end, honest)

LES  = "mass"
AUXC = ["subtlety", "mass_shape", "mass_margins"]
S      = 448 if QUICK else 512
MULT   = 4   if QUICK else 8
EPOCHS = 12  if QUICK else 20
BATCH, LR_HEAD, LR_FT, FREEZE, GAMMA, AUX_W, PATIENCE = 12, 1e-3, 1e-5, 3, 2.0, 0.3, 5

torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
_clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
MEAN = np.array([0.485, 0.456, 0.406], np.float32)
STD  = np.array([0.229, 0.224, 0.225], np.float32)

# ctor, weights, channels, post-activation
SPECS = {
    "convnext_tiny":   (models.convnext_tiny,   models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1,    768, "none"),
    "densenet121":     (models.densenet121,     models.DenseNet121_Weights.IMAGENET1K_V1,     1024, "relu"),
    "efficientnet_b0": (models.efficientnet_b0, models.EfficientNet_B0_Weights.IMAGENET1K_V1, 1280, "none"),
    "resnet50":        (models.resnet50,        models.ResNet50_Weights.IMAGENET1K_V1,        2048, "none"),
}

# ══ PREFLIGHT — real ImageNet weights, or abort before burning hours ══
print("preflight - checking pretrained weights")
bad = []
for name, (fn, w, C, _) in SPECS.items():
    try:
        fn(weights=w); print("  OK   " + name)
    except Exception as e:
        bad.append(name); print("  FAIL " + name + "  -> " + str(e)[:130])
if bad:
    raise SystemExit(
        "\nABORT: no ImageNet weights for: " + ", ".join(bad) +
        "\nRandom init would invalidate the comparison.\n\n"
        "On any machine with internet:\n"
        "  python -c \"from torchvision import models as m; \\\n"
        "    m.convnext_tiny(weights=m.ConvNeXt_Tiny_Weights.IMAGENET1K_V1); \\\n"
        "    m.resnet50(weights=m.ResNet50_Weights.IMAGENET1K_V1); \\\n"
        "    m.efficientnet_b0(weights=m.EfficientNet_B0_Weights.IMAGENET1K_V1)\"\n"
        "then copy ~/.cache/torch/hub/checkpoints/ here.\n"
        "(HF_HUB_OFFLINE does not affect torchvision - it uses its own URL.)")
print("preflight passed\n")

# ══ DATA ══════════════════════════════════════════════════════════════
d = pd.read_csv(os.path.join(D, "unified_folds_" + LES + ".csv")).reset_index(drop=True)
d["label"] = d["label"].astype(int)
PM = os.path.join(D, "predmasks_" + LES)
d["pred"] = d["img"].apply(
    lambda p: os.path.join(PM, os.path.basename(p).replace("_img.png", "") + "_pred.png"))
assert d["pred"].apply(os.path.exists).all(), "missing predicted masks - rerun Phase 2"
print(LES.upper() + "   n=" + str(len(d)) + "   lesions=" + str(d.lesion_key.nunique()) +
      "   patients=" + str(d.patient_id.nunique()))

CACHE = {}; t0 = time.time()
for _, r in d.iterrows():
    k = r["img"]
    if k in CACHE: continue
    im = cv2.resize(cv2.imread(k, cv2.IMREAD_GRAYSCALE), (S, S))
    gt = cv2.imread(r["msk"], cv2.IMREAD_GRAYSCALE)
    gm = (cv2.resize(gt, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.float32) \
         if gt is not None else np.zeros((S, S), np.float32)
    pdd = cv2.imread(r["pred"], cv2.IMREAD_GRAYSCALE)
    pmm = (cv2.resize(pdd, (S, S), interpolation=cv2.INTER_NEAREST) > 127).astype(np.float32) \
          if pdd is not None else np.zeros((S, S), np.float32)
    CACHE[k] = (_clahe.apply(im), gm, pmm)
print("cached " + str(len(CACHE)) + " images in " + format(time.time() - t0, ".0f") + "s")

def primary(x):
    return "UNK" if pd.isna(x) else str(x).split("-")[0].strip().upper()

aux, meta = {}, {}
for c in AUXC:
    if c not in d.columns or d[c].notna().sum() == 0: continue
    if c == "subtlety":
        v = pd.to_numeric(d[c], errors="coerce").where(lambda z: (z >= 1) & (z <= 5))
        codes = (v - 1).fillna(-1).astype(int); n = int(v.max()) if v.notna().any() else 0
    else:
        pr = d[c].map(primary); keep = pr.value_counts().head(6).index.tolist()
        pr = pr.where(pr.isin(keep), "OTHER")
        cats = sorted([q for q in pr.unique() if q != "UNK"]); mp = {q: i for i, q in enumerate(cats)}
        codes = pr.map(lambda z: mp.get(z, -1)).astype(int); n = len(cats)
    if n > 1: aux[c] = codes.values; meta[c] = n
AK = sorted(aux.keys()); print("aux heads:", meta, "\n")

class DS(Dataset):
    def __init__(s, idx, aug, tta=0):
        s.idx = np.array(idx); s.aug = aug; s.m = MULT if aug else 1; s.tta = tta
    def __len__(s): return len(s.idx) * s.m
    def __getitem__(s, i):
        j = s.idx[i % len(s.idx)]; v = i // len(s.idx); r = d.iloc[j]
        img, gm, pm = CACHE[r["img"]]
        mask = (pm if USE_PRED_MASKS else gm).copy(); img = img.copy()
        if s.aug and v > 0:
            if   v == 1: img = np.fliplr(img);   mask = np.fliplr(mask)
            elif v == 2: img = np.flipud(img);   mask = np.flipud(mask)
            elif v == 3: img = np.rot90(img, 1); mask = np.rot90(mask, 1)
            elif v == 4: img = np.rot90(img, 2); mask = np.rot90(mask, 2)
            elif v == 5: img = np.rot90(img, 3); mask = np.rot90(mask, 3)
            elif v == 6:
                M = cv2.getRotationMatrix2D((S/2, S/2), np.random.uniform(-20, 20),
                                            np.random.uniform(.9, 1.1))
                img  = cv2.warpAffine(img,  M, (S, S), borderMode=cv2.BORDER_REFLECT)
                mask = cv2.warpAffine(mask, M, (S, S), flags=cv2.INTER_NEAREST)
            elif v == 7:
                img = np.clip(img.astype(np.float32)*np.random.uniform(.85, 1.15), 0, 255).astype(np.uint8)
        if   s.tta == 1: img = np.fliplr(img);   mask = np.fliplr(mask)
        elif s.tta == 2: img = np.flipud(img);   mask = np.flipud(mask)
        elif s.tta == 3: img = np.rot90(img, 2); mask = np.rot90(mask, 2)
        im = np.ascontiguousarray(img).astype(np.float32) / 255.
        x  = np.stack([im, im, im], 0)
        x  = ((x.transpose(1, 2, 0) - MEAN) / STD).transpose(2, 0, 1).astype(np.float32)
        av = np.array([aux[c][j] for c in AK], dtype=np.int64) if AK else np.zeros(0, np.int64)
        return (torch.from_numpy(np.ascontiguousarray(x)),
                torch.from_numpy(np.ascontiguousarray(mask))[None],
                torch.tensor(int(r["label"])), torch.from_numpy(av))

class Net(nn.Module):
    def __init__(s, bk, mode, meta):
        super().__init__()
        fn, w, C, post = SPECS[bk]
        m = fn(weights=w); s.post = post; s.mode = mode
        s.b = nn.Sequential(*list(m.children())[:-2]) if bk == "resnet50" else m.features
        IN = 2 * C if mode == "dual" else C
        s.head = nn.Sequential(nn.Linear(IN, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, 2))
        s.keys = sorted(meta.keys())
        s.aux  = nn.ModuleList([nn.Sequential(nn.Linear(IN, 128), nn.ReLU(), nn.Dropout(0.3),
                                              nn.Linear(128, meta[k])) for k in s.keys])
    def forward(s, x, mask):
        f = s.b(x)
        if s.post == "relu": f = F.relu(f)
        if s.mode == "dual":
            mm = F.interpolate(mask, size=f.shape[2:], mode="bilinear", align_corners=False)
            w  = 1.0 + 2.0 * mm
            fg = (f * w).sum((2, 3)) / (w.sum((2, 3)) + 1e-6)
            fu = f.mean((2, 3))
            g  = torch.cat([fg, fu], 1)
        else:
            g = f.mean((2, 3))
        return s.head(g), [h(g) for h in s.aux]

y = d["label"].values

def run(bk, mode):
    tag = bk + "_" + mode; f_oof = os.path.join(OUT, tag + ".npy")
    if os.path.exists(f_oof):
        print("  [cached]  " + tag); return np.load(f_oof)
    print("  running   " + tag)
    oof = np.zeros(len(d)); T0 = time.time()
    for k in range(5):
        role = d["role_f" + str(k)]
        tr = np.where(role == "train")[0]; va = np.where(role == "val")[0]; te = np.where(role == "test")[0]
        assert not (set(d.patient_id[tr]) & set(d.patient_id[te])), "PATIENT LEAK"
        assert not (set(d.patient_id[va]) & set(d.patient_id[te])), "PATIENT LEAK"
        t0 = time.time(); torch.manual_seed(k); np.random.seed(k)
        n0 = float((y[tr] == 0).sum()); n1 = float((y[tr] == 1).sum())
        al = torch.tensor([n1/(n0+n1), n0/(n0+n1)], device=DEV)
        def focal(lo, t):
            ce = F.cross_entropy(lo.float(), t, weight=al, reduction="none")
            pt = torch.exp(-ce); return ((1 - pt) ** GAMMA * ce).mean()
        net = Net(bk, mode, meta).to(DEV).to(memory_format=torch.channels_last)
        for p_ in net.b.parameters(): p_.requires_grad = False
        sc  = torch.amp.GradScaler()
        opt = torch.optim.AdamW([p_ for p_ in net.parameters() if p_.requires_grad],
                                lr=LR_HEAD, weight_decay=1e-3)
        tl  = DataLoader(DS(tr, True), batch_size=BATCH, shuffle=True, num_workers=0, pin_memory=True)
        @torch.no_grad()
        def col(idx, tta=True):
            net.eval(); reps = [0, 1, 2, 3] if tta else [0]; tot = None
            for t in reps:
                ld = DataLoader(DS(idx, False, tta=t), batch_size=20, shuffle=False, num_workers=0)
                ps = []
                for x, m, _, _ in ld:
                    x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
                    with torch.amp.autocast(device_type="cuda"): o, _ = net(x, m)
                    ps += list(torch.softmax(o.float(), 1)[:, 1].cpu().numpy())
                ps = np.array(ps); tot = ps if tot is None else tot + ps
            return tot / len(reps)
        best, bs, ni = 0, None, 0
        for ep in range(1, EPOCHS + 1):
            if ep == FREEZE + 1:
                for p_ in net.b.parameters(): p_.requires_grad = True
                opt = torch.optim.AdamW(net.parameters(), lr=LR_FT, weight_decay=1e-3)
            net.train()
            if ep <= FREEZE: net.b.eval()
            for x, m, t2, a in tl:
                x = x.to(DEV).to(memory_format=torch.channels_last); m = m.to(DEV)
                t2 = t2.to(DEV); a = a.to(DEV)
                opt.zero_grad(set_to_none=True)
                with torch.amp.autocast(device_type="cuda"):
                    o, ax = net(x, m); L = focal(o, t2)
                    if len(ax):
                        la = sum(F.cross_entropy(g.float(), a[:, h], ignore_index=-1)
                                 for h, g in enumerate(ax)) / len(ax)
                        L = L + AUX_W * la
                if not torch.isfinite(L): continue
                sc.scale(L).backward(); sc.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(net.parameters(), 5.0)
                sc.step(opt); sc.update()
            pv = col(va, tta=False)
            au = roc_auc_score(y[va], pv) if len(set(y[va])) > 1 else 0
            if au > best:
                best = au; bs = {q: v.cpu().clone() for q, v in net.state_dict().items()}; ni = 0
            else: ni += 1
            if ni >= PATIENCE: break
        net.load_state_dict({q: v.to(DEV) for q, v in bs.items()})
        oof[te] = col(te)
        print("     fold " + str(k) + "  AUC " + format(roc_auc_score(y[te], oof[te]), ".4f") +
              "   (" + format(time.time() - t0, ".0f") + "s)")
        del net; torch.cuda.empty_cache()
    np.save(f_oof, oof)
    print("     -> " + tag + " done in " + format((time.time() - T0) / 60, ".1f") + " min")
    return oof

def per_lesion(p):
    L = d.assign(_p=p).groupby("lesion_key").agg(yy=("label", "max"), pp=("_p", "mean"))
    return roc_auc_score(L.yy, L.pp)

def nparams(bk, mode):
    return sum(p.numel() for p in Net(bk, mode, meta).parameters()) / 1e6

# ══ RUN ═══════════════════════════════════════════════════════════════
R = {}
for bk in SPECS:
    print("\n" + "#" * 70 + "\n#  " + bk + "\n" + "#" * 70)
    for mode in ["blind", "dual"]:
        R[(bk, mode)] = run(bk, mode)

# ══ RESULT TABLE ══════════════════════════════════════════════════════
rows = []
for bk in SPECS:
    rows.append(dict(backbone=bk,
                     params_M=nparams(bk, "dual"),
                     feat=SPECS[bk][2],
                     blind=per_lesion(R[(bk, "blind")]),
                     dual =per_lesion(R[(bk, "dual")])))
for r in rows: r["gain"] = r["dual"] - r["blind"]
pd.DataFrame(rows).to_csv(os.path.join(OUT, "backbone_ablation_summary.csv"), index=False)

print("\n" + "=" * 82)
print("BACKBONE ABLATION  -  " + ("QUICK" if QUICK else "FULL") + " settings,  " +
      ("predicted" if USE_PRED_MASKS else "reference") + " masks,  per-lesion AUC")
print("=" * 82)
print(format("backbone", "<18") + format("params(M)", ">10") + format("feat", ">7") +
      format("blind", ">9") + format("dual", ">9") + format("gain", ">10"))
print("-" * 82)
for r in rows:
    print(format(r["backbone"], "<18") + format(r["params_M"], ">10.2f") +
          format(r["feat"], ">7") + format(r["blind"], ">9.4f") +
          format(r["dual"], ">9.4f") + format(r["gain"], ">+10.4f"))
print("-" * 82)
g = [r["gain"] for r in rows]
print("mean gain " + format(np.mean(g), "+.4f") + "   sd " + format(np.std(g), ".4f") +
      "   min " + format(min(g), "+.4f") + "   max " + format(max(g), "+.4f"))
print("\nIf the gain holds across all four backbones, the mechanism is")
print("backbone-agnostic and not an artefact of DenseNet-121.")
print("=" * 82)

preflight - checking pretrained weights
  OK   convnext_tiny
  OK   densenet121
  OK   efficientnet_b0
  OK   resnet50
preflight passed

MASS   n=1696   lesions=1005   patients=892
cached 1696 images in 11s
aux heads: {'subtlety': 5, 'mass_shape': 7, 'mass_margins': 5} 


######################################################################
#  convnext_tiny
######################################################################
  running   convnext_tiny_blind
     fold 0  AUC 0.7678   (492s)
     fold 1  AUC 0.8702   (408s)
     fold 2  AUC 0.8153   (355s)
     fold 3  AUC 0.8522   (412s)
     fold 4  AUC 0.8651   (380s)
     -> convnext_tiny_blind done in 34.1 min
  running   convnext_tiny_dual
     fold 0  AUC 0.7865   (416s)
     fold 1  AUC 0.8618   (409s)
     fold 2  AUC 0.8181   (411s)
     fold 3  AUC 0.8423   (422s)
     fold 4  AUC 0.8586   (344s)
     -> convnext_tiny_dual done in 33.4 min

######################################################################
#  densenet121

In [1]:
# ======================================================================
#  Cell A2 — one derivation, every number
#
#  Part B compares three ways of pooling out-of-fold scores and prints
#  the ensemble AUC for each. Set MODE to whichever reproduces Cell 35,
#  re-run, and Parts C-E give you the ablation, the decision policy and
#  the per-BI-RADS table all from that one choice.
#
#  CPU only. ~2 minutes.
# ======================================================================
import os, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

ROOT      = "/root/autodl-tmp/CBIS"
MODE      = "rank_global"   # "prob" | "rank_local" | "rank_global"
SENS_FLOOR = 0.70           # cohort-level sensitivity constraint
LEAK_AUC  = 0.97
N_BOOT    = 1000
MIN_N     = 25
rng       = np.random.default_rng(0)

ALLOW = ["cv_mass_v2", "cv_mass_efficientnet_b0", "cv_mass_handcrafted",
         "cv_mass_endtoend_fused", "cv_mass_twostream",
         "cv_mass_twostream_calcpre", "cv_mass_imageonly"]
WIDE  = ["cv_mass_twostream", "cv_mass_twostream_calcpre"]

key = lambda p: os.path.splitext(os.path.basename(str(p)))[0]

# ============ PART A — load ==========================================
M = pd.read_csv(f"{ROOT}/unified_folds_mass.csv")
M["_k"] = M["img"].map(key)
M["_y"] = M["pathology"].astype(str).str.upper().str.contains("MALIGNANT").astype(int)
M["_fold"] = -1
for k in range(5):
    M.loc[M[f"role_f{k}"].astype(str).str.lower().eq("test"), "_fold"] = k
assert (M["_fold"] >= 0).all() and M["_k"].is_unique
K2L = dict(zip(M["_k"], M["lesion_key"]))

cols = {}
for name in ALLOW:
    d = pd.read_csv(f"{ROOT}/{name}_oof.csv")
    d["_k"] = d["img"].map(key)
    assert d["_k"].isin(K2L).mean() > 0.99, f"{name}: img values do not match"
    d["lesion_key"] = d["_k"].map(K2L)
    cols[name] = d.groupby("lesion_key")["prob"].mean().rename(name)

L = (M.groupby("lesion_key")
       .agg(_y=("_y","max"), _fold=("_fold","max"), subtlety=("subtlety","first"),
            shape=("mass_shape","first"), margin=("mass_margins","first"),
            birads=("assessment","first")))
X = L.join(pd.concat(cols.values(), axis=1), how="inner").dropna(subset=ALLOW)
assert len(X) > 900, f"coverage collapsed to {len(X)}"

Y  = X["_y"].to_numpy()
FD = X["_fold"].to_numpy().astype(int)
BR = X["birads"].fillna(-1).astype(int).to_numpy()
V  = {m: X[m].to_numpy() for m in ALLOW}
print(f"loaded   : {len(X)} lesions, {Y.mean():.1%} malignant, "
      f"BI-RADS {sorted(set(BR))}")

print("\nper-model out-of-fold AUC")
for m in ALLOW:
    a = roc_auc_score(Y, V[m])
    assert a < LEAK_AUC, f"leakage guard tripped on {m} (AUC {a:.4f})"
    print(f"  {m:<28s} {a:.4f}")

# ============ PART B — how should scores be pooled? ==================
rank01 = lambda v: (rankdata(v) - 1) / max(len(v) - 1, 1)
G = {m: rank01(V[m]) for m in ALLOW}          # global ranks, computed once

def combine(sub, idx, mode):
    if mode == "prob":        src = V
    elif mode == "rank_global": src = G
    elif mode == "rank_local":
        return np.column_stack([rank01(V[m][idx]) for m in sub]).mean(1)
    else: raise ValueError(mode)
    return np.column_stack([src[m][idx] for m in sub]).mean(1)

def greedy(pool, idx, mode):
    chosen, best = [], -np.inf
    while True:
        pm, pa = None, best
        for m in pool:
            if m in chosen: continue
            a = roc_auc_score(Y[idx], combine(chosen + [m], idx, mode))
            if a > pa + 1e-6: pm, pa = m, a
        if pm is None: return chosen
        chosen.append(pm); best = pa

def build(pool, mode, verbose=False):
    out, picks = np.full(len(Y), np.nan), {}
    for k in sorted(set(FD)):
        te, tr = np.where(FD == k)[0], np.where(FD != k)[0]
        sel = greedy(pool, tr, mode); picks[k] = sel
        out[te] = combine(sel, te, mode)
        if verbose: print(f"    fold {k}: {' + '.join(sel)}")
    return out, picks

print("\n" + "=" * 70)
print("PART B  pooling comparison  (which one is Cell 35?)")
print("=" * 70)
for mode in ("prob", "rank_local", "rank_global"):
    s, _ = build(ALLOW, mode)
    flag = "   <-- MODE in use" if mode == MODE else ""
    print(f"  {mode:<12s} pooled AUC = {roc_auc_score(Y, s):.4f}{flag}")
print("\n  If one of these is 0.9078, set MODE to it at the top and re-run.")

# ============ PART C — ablation ======================================
print("\n" + "=" * 70)
print(f"PART C  wide-context ablation   (MODE = {MODE})")
print("=" * 70)
print("  FULL:")
full, picks = build(ALLOW, MODE, verbose=True)
print("  NO-WIDE:")
nowd, _ = build([m for m in ALLOW if m not in WIDE], MODE, verbose=True)
auc_f, auc_n = roc_auc_score(Y, full), roc_auc_score(Y, nowd)
print(f"\n  FULL {auc_f:.4f}   NO-WIDE {auc_n:.4f}   delta {auc_f-auc_n:+.4f}")

groups = [("all lesions", np.ones(len(Y), bool))]
for col in ("subtlety", "shape", "margin"):
    for v in sorted(X[col].dropna().astype(str).unique()):
        msk = X[col].astype(str).eq(v).to_numpy()
        if msk.sum() >= MIN_N: groups.append((f"{col} = {v.lower()}", msk))

rows = []
for name, msk in groups:
    yy = Y[msk]
    if yy.min() == yy.max(): continue
    a_f, a_n = roc_auc_score(yy, full[msk]), roc_auc_score(yy, nowd[msk])
    idx, bs = np.where(msk)[0], []
    for _ in range(N_BOOT):
        r = rng.choice(idx, size=len(idx), replace=True)
        if Y[r].min() != Y[r].max():
            bs.append(roc_auc_score(Y[r], full[r]) - roc_auc_score(Y[r], nowd[r]))
    lo, hi = np.percentile(bs, [2.5, 97.5])
    rows.append({"subgroup": name, "n": int(msk.sum()), "malig": f"{yy.mean():.0%}",
                 "with": round(a_f,4), "without": round(a_n,4),
                 "delta": round(a_f-a_n,4), "CI95": f"[{lo:+.3f}, {hi:+.3f}]",
                 "sig": "yes" if (lo > 0 or hi < 0) else ""})
ABL = pd.DataFrame(rows).sort_values("delta", ascending=False)
pd.set_option("display.width", 170, "display.max_rows", 90)
print("\n" + ABL.to_string(index=False))

# ============ PART D — decision policy ===============================
def metrics(y, pred):
    tp = int(((pred == 1) & (y == 1)).sum()); fn = int(((pred == 0) & (y == 1)).sum())
    tn = int(((pred == 0) & (y == 0)).sum()); fp = int(((pred == 1) & (y == 0)).sum())
    return dict(acc=(tp+tn)/len(y), sens=tp/max(tp+fn,1), spec=tn/max(tn+fp,1),
                FP=fp, FN=fn)

GRID = np.linspace(0.005, 0.995, 199)

def fit_global(y, s):
    return max(GRID, key=lambda t: ((s >= t).astype(int) == y).mean())

def apply_joint(s, b, thr):
    return np.array([int(si >= thr.get(bi, thr["_"])) for si, bi in zip(s, b)])

def fit_joint(y, s, b, floor):
    g = fit_global(y, s)
    thr = {"_": g}; thr.update({c: g for c in sorted(set(b))})
    best = metrics(y, apply_joint(s, b, thr))["acc"]
    for _ in range(6):
        moved = False
        for c in sorted(set(b)):
            for t in GRID:
                trial = dict(thr); trial[c] = t
                mm = metrics(y, apply_joint(s, b, trial))
                if mm["sens"] >= floor and mm["acc"] > best + 1e-9:
                    thr, best, moved = trial, mm["acc"], True
        if not moved: break
    return thr

pred_g = np.zeros(len(Y), int); pred_j = np.zeros(len(Y), int)
for k in sorted(set(FD)):
    te, tr = np.where(FD == k)[0], np.where(FD != k)[0]
    pred_g[te] = (full[te] >= fit_global(Y[tr], full[tr])).astype(int)
    pred_j[te] = apply_joint(full[te], BR[te], fit_joint(Y[tr], full[tr], BR[tr], SENS_FLOOR))

print("\n" + "=" * 70)
print("PART D  decision policy   (thresholds fitted on inner folds only)")
print("=" * 70)
print(f"  ensemble AUC                 {auc_f:.4f}")
for tag, pr in [("autonomous: global threshold", pred_g),
                (f"assisted: joint per-BI-RADS (sens floor {SENS_FLOOR})", pred_j)]:
    m = metrics(Y, pr)
    print(f"  {tag:<44s} acc {m['acc']:.1%}  sens {m['sens']:.3f}  "
          f"spec {m['spec']:.3f}  FP {m['FP']}  FN {m['FN']}")

# ============ PART E — per-BI-RADS table =============================
out = []
for c in sorted(set(BR)):
    msk = BR == c; yy = Y[msk]
    a = roc_auc_score(yy, full[msk]) if yy.min() != yy.max() else np.nan
    m = metrics(yy, pred_j[msk])
    out.append({"BI-RADS": c, "n": int(msk.sum()), "malig": int(yy.sum()),
                "benign": int((1-yy).sum()), "AUC": round(a,4) if a==a else "n/a",
                "acc": f"{m['acc']:.1%}", "sens": round(m['sens'],3),
                "FP": m["FP"], "missed": m["FN"],
                "err_share": f"{(m['FP']+m['FN'])/max((pred_j!=Y).sum(),1):.0%}"})
PB = pd.DataFrame(out)
print("\n" + "=" * 70)
print("PART E  per-BI-RADS at the assisted operating point")
print("=" * 70)
print(PB.to_string(index=False))

ABL.to_csv(f"{ROOT}/ablation_twostream_clean.csv", index=False)
PB.to_csv(f"{ROOT}/birads_table_clean.csv", index=False)
print(f"\nsaved -> ablation_twostream_clean.csv, birads_table_clean.csv")

loaded   : 1005 lesions, 46.0% malignant, BI-RADS [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5)]

per-model out-of-fold AUC
  cv_mass_v2                   0.8692
  cv_mass_efficientnet_b0      0.8536
  cv_mass_handcrafted          0.6963
  cv_mass_endtoend_fused       0.8773
  cv_mass_twostream            0.8885
  cv_mass_twostream_calcpre    0.8656
  cv_mass_imageonly            0.7796

PART B  pooling comparison  (which one is Cell 35?)
  prob         pooled AUC = 0.9110
  rank_local   pooled AUC = 0.9137
  rank_global  pooled AUC = 0.9090   <-- MODE in use

  If one of these is 0.9078, set MODE to it at the top and re-run.

PART C  wide-context ablation   (MODE = rank_global)
  FULL:
    fold 0: cv_mass_twostream + cv_mass_endtoend_fused + cv_mass_twostream_calcpre
    fold 1: cv_mass_twostream + cv_mass_endtoend_fused + cv_mass_twostream_calcpre
    fold 2: cv_mass_twostream + cv_mass_endtoend_fused + cv_mass_twostream_calcpre
    fold 3: cv_mass_two

In [7]:
# ======================================================================
#  FIGURE CELL 1  ->  paper_fig2_examples.png
#  Figure 2:  benign mass + its mask,  malignant mass + its mask
#  Standalone. No GPU. No training. ~15 seconds.
# ======================================================================
import os, glob, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT, DPI, PICK = ".", 300, 0          # PICK: 0 = first usable lesion, 1 = next, ...

# ── 1. the fold table ─────────────────────────────────────────────────
cand = glob.glob("**/unified_folds_mass.csv", recursive=True)
assert cand, "unified_folds_mass.csv not found - run the PHASE 1 cell first"
CSV = cand[0]; df = pd.read_csv(CSV)
print("fold table :", CSV, "|", len(df), "rows")

# ── 2. index every JPEG on disk by its DICOM series UID ───────────────
#    The CSV stores .dcm paths, but the pixels live in  jpeg/<series-uid>/*.jpg
ROOTS = {".", os.path.dirname(CSV) or ".",
         os.path.dirname(os.path.dirname(CSV)) or "."}
JPG = {}
for rt in ROOTS:
    for p in glob.glob(os.path.join(rt, "**", "jpeg", "*", "*.jpg"), recursive=True):
        JPG.setdefault(os.path.basename(os.path.dirname(p)), set()).add(p)
JPG = {k: sorted(v) for k, v in JPG.items()}
print("JPEG series:", len(JPG), "indexed")
assert JPG, "no  jpeg/<series-uid>/*.jpg  tree found under " + str(ROOTS)

def _uid(v):
    """last DICOM series UID in a path (the folder that holds the .dcm)"""
    for c in reversed([c for c in str(v).replace("\\", "/").split("/") if c]):
        if c.count(".") >= 6:
            return c
    return None

def _binfrac(p):
    """~1.0 for a binary ROI mask, ~0.2-0.7 for a real mammogram"""
    a = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if a is None: return -1.0
    s = cv2.resize(a, (256, 256))
    return float(((s > 200) | (s < 40)).mean())

def _cands(row, cols):
    out = []
    for c in cols:
        v = row.get(c)
        if not isinstance(v, str): continue
        v = v.strip()
        if os.path.exists(v) and v.lower().endswith((".png", ".jpg", ".jpeg")):
            out.append(v)
        u = v if v in JPG else _uid(v)
        if u in JPG:
            out += JPG[u]
    return list(dict.fromkeys(out))

IMG_COLS = ["img", "full_path", "full_series", "cropped image file path"]
MSK_COLS = ["msk", "mask_path", "mask_series"]

def resolve_pair(row):
    """A CBIS 'ROI mask images' series holds BOTH a crop and a full-size mask.
       Pick the mask by binariness, then the image whose shape matches it."""
    mcs = _cands(row, MSK_COLS)
    if not mcs: return None
    mp = max(mcs, key=_binfrac)
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if m is None or _binfrac(mp) < 0.90: return None
    for p in _cands(row, IMG_COLS):
        if p == mp: continue
        a = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if a is not None and a.shape == m.shape:
            return p, mp
    return None

def usable(sub, k=0):
    seen = 0
    for _, row in sub.iterrows():
        pr = resolve_pair(row)
        if pr:
            if seen == k: return row, pr[0], pr[1]
            seen += 1
    raise RuntimeError("no lesion resolved - lower PICK, or the jpeg/ tree is incomplete")

def crop_to_lesion(img, msk, size=512, pad=0.35):
    ys, xs = np.where(msk > 127)
    if len(ys) == 0:
        return cv2.resize(img, (size, size)), cv2.resize(msk, (size, size))
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    h = int(max(y1 - y0, x1 - x0) * (1 + pad)) // 2 + 1
    Y0, Y1 = max(0, cy - h), min(img.shape[0], cy + h)
    X0, X1 = max(0, cx - h), min(img.shape[1], cx + h)
    return (cv2.resize(img[Y0:Y1, X0:X1], (size, size)),
            cv2.resize(msk[Y0:Y1, X0:X1], (size, size),
                       interpolation=cv2.INTER_NEAREST))

def get(sub, k=0):
    row, ip, mp = usable(sub, k)
    print("  image:", os.path.relpath(ip))
    print("  mask :", os.path.relpath(mp))
    return crop_to_lesion(cv2.imread(ip, cv2.IMREAD_GRAYSCALE),
                          cv2.imread(mp, cv2.IMREAD_GRAYSCALE))

plt.rcParams.update({"font.family": "serif",
                     "font.serif": ["Times New Roman", "Liberation Serif",
                                    "DejaVu Serif"]})

def panel(ax, arr, title, fs=9.5):
    ax.imshow(arr, cmap="gray" if arr.ndim == 2 else None)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor("#999999"); s.set_linewidth(0.8)
    ax.set_title(title, fontsize=fs, pad=4)

# ── Figure 2 ──────────────────────────────────────────────────────────
C_LAB = "label" if "label" in df.columns else "pathology_bin"

fig, axes = plt.subplots(1, 4, figsize=(11.0, 3.1))
for j, (lab, name) in enumerate([(0, "Benign"), (1, "Malignant")]):
    print(name + ":")
    img, msk = get(df[df[C_LAB] == lab], PICK)
    panel(axes[2 * j],     img, name + " mass")
    panel(axes[2 * j + 1], msk, name + " reference mask")

fig.tight_layout()
dst = os.path.abspath(os.path.join(OUT, "paper_fig2_examples.png"))
fig.savefig(dst, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("\nSAVED TO:", dst)
print("this is the FIGURE SLOT for Figure 2")

fold table : CBIS/unified_folds_mass.csv | 1696 rows
JPEG series: 6774 indexed
Benign:
  image: CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.215081818713600536113960661873725083371/1-103.jpg
  mask : CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.199593071810497070809647901570077988031/2-087.jpg
Malignant:
  image: CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172/1-271.jpg
  mask : CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.30820586311062570442302321942433426184/1-083.jpg

SAVED TO: /root/autodl-tmp/paper_fig2_examples.png
this is the FIGURE SLOT for Figure 2


In [8]:
# ======================================================================
#  FIGURE CELL 2  ->  paper_fig3_preprocessing.png
#  Figure 3:  original -> CLAHE -> binarised mask -> boundary overlay
#  Standalone. No GPU. No training. ~15 seconds.
# ======================================================================
import os, glob, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT, DPI, PICK = ".", 300, 0          # PICK: 0 = first usable lesion, 1 = next, ...

# ── 1. the fold table ─────────────────────────────────────────────────
cand = glob.glob("**/unified_folds_mass.csv", recursive=True)
assert cand, "unified_folds_mass.csv not found - run the PHASE 1 cell first"
CSV = cand[0]; df = pd.read_csv(CSV)
print("fold table :", CSV, "|", len(df), "rows")

# ── 2. index every JPEG on disk by its DICOM series UID ───────────────
#    The CSV stores .dcm paths, but the pixels live in  jpeg/<series-uid>/*.jpg
ROOTS = {".", os.path.dirname(CSV) or ".",
         os.path.dirname(os.path.dirname(CSV)) or "."}
JPG = {}
for rt in ROOTS:
    for p in glob.glob(os.path.join(rt, "**", "jpeg", "*", "*.jpg"), recursive=True):
        JPG.setdefault(os.path.basename(os.path.dirname(p)), set()).add(p)
JPG = {k: sorted(v) for k, v in JPG.items()}
print("JPEG series:", len(JPG), "indexed")
assert JPG, "no  jpeg/<series-uid>/*.jpg  tree found under " + str(ROOTS)

def _uid(v):
    """last DICOM series UID in a path (the folder that holds the .dcm)"""
    for c in reversed([c for c in str(v).replace("\\", "/").split("/") if c]):
        if c.count(".") >= 6:
            return c
    return None

def _binfrac(p):
    """~1.0 for a binary ROI mask, ~0.2-0.7 for a real mammogram"""
    a = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if a is None: return -1.0
    s = cv2.resize(a, (256, 256))
    return float(((s > 200) | (s < 40)).mean())

def _cands(row, cols):
    out = []
    for c in cols:
        v = row.get(c)
        if not isinstance(v, str): continue
        v = v.strip()
        if os.path.exists(v) and v.lower().endswith((".png", ".jpg", ".jpeg")):
            out.append(v)
        u = v if v in JPG else _uid(v)
        if u in JPG:
            out += JPG[u]
    return list(dict.fromkeys(out))

IMG_COLS = ["img", "full_path", "full_series", "cropped image file path"]
MSK_COLS = ["msk", "mask_path", "mask_series"]

def resolve_pair(row):
    """A CBIS 'ROI mask images' series holds BOTH a crop and a full-size mask.
       Pick the mask by binariness, then the image whose shape matches it."""
    mcs = _cands(row, MSK_COLS)
    if not mcs: return None
    mp = max(mcs, key=_binfrac)
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if m is None or _binfrac(mp) < 0.90: return None
    for p in _cands(row, IMG_COLS):
        if p == mp: continue
        a = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if a is not None and a.shape == m.shape:
            return p, mp
    return None

def usable(sub, k=0):
    seen = 0
    for _, row in sub.iterrows():
        pr = resolve_pair(row)
        if pr:
            if seen == k: return row, pr[0], pr[1]
            seen += 1
    raise RuntimeError("no lesion resolved - lower PICK, or the jpeg/ tree is incomplete")

def crop_to_lesion(img, msk, size=512, pad=0.35):
    ys, xs = np.where(msk > 127)
    if len(ys) == 0:
        return cv2.resize(img, (size, size)), cv2.resize(msk, (size, size))
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    h = int(max(y1 - y0, x1 - x0) * (1 + pad)) // 2 + 1
    Y0, Y1 = max(0, cy - h), min(img.shape[0], cy + h)
    X0, X1 = max(0, cx - h), min(img.shape[1], cx + h)
    return (cv2.resize(img[Y0:Y1, X0:X1], (size, size)),
            cv2.resize(msk[Y0:Y1, X0:X1], (size, size),
                       interpolation=cv2.INTER_NEAREST))

def get(sub, k=0):
    row, ip, mp = usable(sub, k)
    print("  image:", os.path.relpath(ip))
    print("  mask :", os.path.relpath(mp))
    return crop_to_lesion(cv2.imread(ip, cv2.IMREAD_GRAYSCALE),
                          cv2.imread(mp, cv2.IMREAD_GRAYSCALE))

plt.rcParams.update({"font.family": "serif",
                     "font.serif": ["Times New Roman", "Liberation Serif",
                                    "DejaVu Serif"]})

def panel(ax, arr, title, fs=9.5):
    ax.imshow(arr, cmap="gray" if arr.ndim == 2 else None)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor("#999999"); s.set_linewidth(0.8)
    ax.set_title(title, fontsize=fs, pad=4)

# ── Figure 3 ──────────────────────────────────────────────────────────
C_LAB = "label" if "label" in df.columns else "pathology_bin"

print("Malignant example:")
img, msk = get(df[df[C_LAB] == 1], PICK)

enh  = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)  # pipeline setting
binm = (msk > 127).astype(np.uint8) * 255                              # pipeline threshold

ovl = cv2.cvtColor(enh, cv2.COLOR_GRAY2RGB)
cnts, _ = cv2.findContours(binm, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
cv2.drawContours(ovl, cnts, -1, (255, 210, 0), 3)

fig, axes = plt.subplots(1, 4, figsize=(11.0, 3.1))
panel(axes[0], img,  "(a)  original crop")
panel(axes[1], enh,  "(b)  CLAHE, clip 2.0, 8x8 tiles")
panel(axes[2], binm, "(c)  binarised mask, t = 127")
panel(axes[3], ovl,  "(d)  boundary on enhanced crop")

fig.tight_layout()
dst = os.path.abspath(os.path.join(OUT, "paper_fig3_preprocessing.png"))
fig.savefig(dst, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("\nSAVED TO:", dst)
print("this is the FIGURE SLOT for Figure 3")


fold table : CBIS/unified_folds_mass.csv | 1696 rows
JPEG series: 6774 indexed
Malignant example:
  image: CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172/1-271.jpg
  mask : CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.30820586311062570442302321942433426184/1-083.jpg

SAVED TO: /root/autodl-tmp/paper_fig3_preprocessing.png
this is the FIGURE SLOT for Figure 3


In [9]:
# ======================================================================
#  FIGURE CELL 3  ->  paper_fig4_augmentation.png
#  Figure 4:  the eight deterministic Stage-1 transforms
#  Standalone. No GPU. No training. ~15 seconds.
# ======================================================================
import os, glob, numpy as np, pandas as pd, cv2
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt

OUT, DPI, PICK = ".", 300, 0          # PICK: 0 = first usable lesion, 1 = next, ...

# ── 1. the fold table ─────────────────────────────────────────────────
cand = glob.glob("**/unified_folds_mass.csv", recursive=True)
assert cand, "unified_folds_mass.csv not found - run the PHASE 1 cell first"
CSV = cand[0]; df = pd.read_csv(CSV)
print("fold table :", CSV, "|", len(df), "rows")

# ── 2. index every JPEG on disk by its DICOM series UID ───────────────
#    The CSV stores .dcm paths, but the pixels live in  jpeg/<series-uid>/*.jpg
ROOTS = {".", os.path.dirname(CSV) or ".",
         os.path.dirname(os.path.dirname(CSV)) or "."}
JPG = {}
for rt in ROOTS:
    for p in glob.glob(os.path.join(rt, "**", "jpeg", "*", "*.jpg"), recursive=True):
        JPG.setdefault(os.path.basename(os.path.dirname(p)), set()).add(p)
JPG = {k: sorted(v) for k, v in JPG.items()}
print("JPEG series:", len(JPG), "indexed")
assert JPG, "no  jpeg/<series-uid>/*.jpg  tree found under " + str(ROOTS)

def _uid(v):
    """last DICOM series UID in a path (the folder that holds the .dcm)"""
    for c in reversed([c for c in str(v).replace("\\", "/").split("/") if c]):
        if c.count(".") >= 6:
            return c
    return None

def _binfrac(p):
    """~1.0 for a binary ROI mask, ~0.2-0.7 for a real mammogram"""
    a = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
    if a is None: return -1.0
    s = cv2.resize(a, (256, 256))
    return float(((s > 200) | (s < 40)).mean())

def _cands(row, cols):
    out = []
    for c in cols:
        v = row.get(c)
        if not isinstance(v, str): continue
        v = v.strip()
        if os.path.exists(v) and v.lower().endswith((".png", ".jpg", ".jpeg")):
            out.append(v)
        u = v if v in JPG else _uid(v)
        if u in JPG:
            out += JPG[u]
    return list(dict.fromkeys(out))

IMG_COLS = ["img", "full_path", "full_series", "cropped image file path"]
MSK_COLS = ["msk", "mask_path", "mask_series"]

def resolve_pair(row):
    """A CBIS 'ROI mask images' series holds BOTH a crop and a full-size mask.
       Pick the mask by binariness, then the image whose shape matches it."""
    mcs = _cands(row, MSK_COLS)
    if not mcs: return None
    mp = max(mcs, key=_binfrac)
    m = cv2.imread(mp, cv2.IMREAD_GRAYSCALE)
    if m is None or _binfrac(mp) < 0.90: return None
    for p in _cands(row, IMG_COLS):
        if p == mp: continue
        a = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if a is not None and a.shape == m.shape:
            return p, mp
    return None

def usable(sub, k=0):
    seen = 0
    for _, row in sub.iterrows():
        pr = resolve_pair(row)
        if pr:
            if seen == k: return row, pr[0], pr[1]
            seen += 1
    raise RuntimeError("no lesion resolved - lower PICK, or the jpeg/ tree is incomplete")

def crop_to_lesion(img, msk, size=512, pad=0.35):
    ys, xs = np.where(msk > 127)
    if len(ys) == 0:
        return cv2.resize(img, (size, size)), cv2.resize(msk, (size, size))
    y0, y1, x0, x1 = ys.min(), ys.max(), xs.min(), xs.max()
    cy, cx = (y0 + y1) // 2, (x0 + x1) // 2
    h = int(max(y1 - y0, x1 - x0) * (1 + pad)) // 2 + 1
    Y0, Y1 = max(0, cy - h), min(img.shape[0], cy + h)
    X0, X1 = max(0, cx - h), min(img.shape[1], cx + h)
    return (cv2.resize(img[Y0:Y1, X0:X1], (size, size)),
            cv2.resize(msk[Y0:Y1, X0:X1], (size, size),
                       interpolation=cv2.INTER_NEAREST))

def get(sub, k=0):
    row, ip, mp = usable(sub, k)
    print("  image:", os.path.relpath(ip))
    print("  mask :", os.path.relpath(mp))
    return crop_to_lesion(cv2.imread(ip, cv2.IMREAD_GRAYSCALE),
                          cv2.imread(mp, cv2.IMREAD_GRAYSCALE))

plt.rcParams.update({"font.family": "serif",
                     "font.serif": ["Times New Roman", "Liberation Serif",
                                    "DejaVu Serif"]})

def panel(ax, arr, title, fs=9.5):
    ax.imshow(arr, cmap="gray" if arr.ndim == 2 else None)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_edgecolor("#999999"); s.set_linewidth(0.8)
    ax.set_title(title, fontsize=fs, pad=4)

# ── Figure 4 ──────────────────────────────────────────────────────────
C_LAB = "label" if "label" in df.columns else "pathology_bin"

print("Malignant example:")
img, msk = get(df[df[C_LAB] == 1], PICK)
enh = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(img)

def dihedral(a, k, flip):
    a = np.rot90(a, k)
    return np.fliplr(a) if flip else a

lab = ["0", "90", "180", "270"]
fig, axes = plt.subplots(1, 8, figsize=(15.6, 2.35))
for i, (k, f) in enumerate([(k, f) for f in (0, 1) for k in range(4)]):
    panel(axes[i], dihedral(enh, k, f),
          lab[k] + "\u00b0" + (" + flip" if f else ""), fs=9.0)

fig.tight_layout()
dst = os.path.abspath(os.path.join(OUT, "paper_fig4_augmentation.png"))
fig.savefig(dst, dpi=DPI, bbox_inches="tight", facecolor="white")
plt.close(fig)
print("\nSAVED TO:", dst)
print("this is the FIGURE SLOT for Figure 4")


fold table : CBIS/unified_folds_mass.csv | 1696 rows
JPEG series: 6774 indexed
Malignant example:
  image: CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172/1-271.jpg
  mask : CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.30820586311062570442302321942433426184/1-083.jpg

SAVED TO: /root/autodl-tmp/paper_fig4_augmentation.png
this is the FIGURE SLOT for Figure 4


In [1]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 20 — FULL METRICS FOR EVERY BASELINE, SINGLE GLOBAL THRESHOLD
#   Threshold fitted on official TRAIN, frozen, applied to official TEST.
#   Two variants: at the 0.90 sensitivity floor (your protocol), and
#   unconstrained maximum accuracy (what most papers report).
#   No GPU, no training. ~1 min.
# ══════════════════════════════════════════════════════════════════════════
import os, glob, numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score
D=r"/root/autodl-tmp/CBIS"; stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]
GRID=np.round(np.arange(0.01,0.995,0.005),3)

d=pd.read_csv(os.path.join(D,"unified_folds_mass.csv")); d["_k"]=d["img"].map(stem)
if "side" not in d.columns:
    fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv")); fx["_k"]=fx["cropped image file path"].map(stem)
    d=d.merge(fx[["_k","side"]].drop_duplicates("_k"),on="_k",how="left")
d["y"]=d.label.astype(int)
d["breast_key"]=d.patient_id.astype(str)+"_"+d["side"].astype(str)
d["sp"]=np.where(d.official_split.astype(str).str.lower().str.contains("test"),"test","train")
B=d.set_index("_k")
LEVELS=[("ROI",None),("LESION","lesion_key"),("BREAST","breast_key"),("PATIENT","patient_id")]

def load(fn):
    f=os.path.join(D,fn)
    if not os.path.exists(f): return None
    m=pd.read_csv(f)
    if "img" not in m.columns: return None
    pc="prob" if "prob" in m.columns else None
    if pc is None:
        c=[x for x in m.columns if m[x].dtype.kind=="f" and m[x].between(0,1).all()]
        if not c: return None
        pc=c[0]
    m["_k"]=m["img"].map(stem)
    m=m[m["_k"].isin(B.index)].drop_duplicates("_k")
    return m.set_index("_k")[pc]

def agg(s,key):
    t=pd.DataFrame(dict(p=s.values,y=B.loc[s.index,"y"].values,
                        k=(s.index.values if key is None else B.loc[s.index,key].values)))
    g=t.groupby("k").agg(p=("p","mean"),y=("y","max"))
    return g.y.values.astype(int), g.p.values

def _cut(pos,neg,g):
    return (len(pos)-np.searchsorted(pos,g,"left")).astype(float), \
           (len(neg)-np.searchsorted(neg,g,"left")).astype(float)
def fit_thr(y,p,floor=None):
    P,N=int(y.sum()),len(y)
    tp,fp=_cut(np.sort(p[y==1]),np.sort(p[y==0]),GRID)
    acc,se=(tp+(N-P)-fp)/N, tp/max(P,1)
    if floor is None: return float(GRID[int(np.argmax(acc))])
    ok=se>=floor-1e-12
    return float(GRID[int(np.argmax(np.where(ok,acc,-1.0)))]) if ok.any() else float(GRID[0])
def M(y,p,thr):
    yh=(p>=thr).astype(int)
    tp=int(((yh==1)&(y==1)).sum()); fn=int(((yh==0)&(y==1)).sum())
    tn=int(((yh==0)&(y==0)).sum()); fp=int(((yh==1)&(y==0)).sum())
    return roc_auc_score(y,p),(tp+tn)/len(y),tp/max(tp+fn,1),tn/max(tn+fp,1),fn

PAIRS=[]
for trf in sorted(glob.glob(os.path.join(D,"*officialtrain*.csv"))):
    b=os.path.basename(trf); tef=b.replace("officialtrain","officialsplit")
    A,C=load(b),load(tef)
    if A is None or C is None: continue
    A=A[B.loc[A.index,"sp"].eq("train").values]; C=C[B.loc[C.index,"sp"].eq("test").values]
    if len(A)<900 or len(C)<300: continue
    PAIRS.append((b.replace("cv_mass_","").replace("_officialtrain_oof","").replace(".csv",""),A,C))
print("models with both train and test predictions: %d\n" % len(PAIRS))

for FLOOR,tag in ((0.90,"SENSITIVITY FLOOR 0.90  (your protocol)"),
                  (None,"UNCONSTRAINED MAX ACCURACY  (what most papers report)")):
    print("="*100); print(tag); print("="*100)
    for nm,key in LEVELS:
        print("\n  %s LEVEL" % nm)
        print("    %-30s %-8s %-8s %-7s %-7s %-7s %s"
              % ("model","AUC","acc","sens","spec","thresh","missed"))
        out=[]
        for name,A,C in PAIRS:
            ytr,ptr=agg(A,key); yte,pte=agg(C,key)
            t=fit_thr(ytr,ptr,FLOOR)
            au,ac,se,sp,fn=M(yte,pte,t)
            out.append((au,name,ac,se,sp,t,fn))
        for au,name,ac,se,sp,t,fn in sorted(out):
            star="  <== YOURS" if name.startswith("twostream") and "calc" not in name else ""
            print("    %-30s %-8.4f %-8.4f %-7.3f %-7.3f %-7.3f %-6d%s"
                  % (name,au,ac,se,sp,t,fn,star))

models with both train and test predictions: 8

SENSITIVITY FLOOR 0.90  (your protocol)

  ROI LEVEL
    model                          AUC      acc      sens    spec    thresh  missed
    handcrafted                    0.6781   0.4894   0.891   0.234   0.300   16    
    imageonly                      0.7994   0.5820   0.952   0.346   0.360   7     
    endtoend_fused                 0.8192   0.6561   0.864   0.524   0.300   20    
    endtoend                       0.8194   0.6587   0.898   0.506   0.415   15    
    twostream_calcpre              0.8582   0.7143   0.932   0.576   0.435   10    
    twostream_calcpre_cvmasks      0.8660   0.7328   0.918   0.615   0.435   12    
    twostream                      0.8769   0.7566   0.884   0.675   0.440   17      <== YOURS
    twostream_1seed                0.8769   0.7566   0.884   0.675   0.440   17      <== YOURS

  LESION LEVEL
    model                          AUC      acc      sens    spec    thresh  missed
    handcrafted      

In [1]:
# Run Petrini's model on your test set
# ══════════════════════════════════════════════════════════════════════════
# CELL 22 — BUILD CC + MLO FULL-MAMMOGRAM PAIRS FOR THE OFFICIAL TEST SET
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
D=r"/root/autodl-tmp/CBIS"
stem=lambda p: os.path.splitext(os.path.basename(str(p)))[0]

fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv"))
print("columns:", list(fx.columns))
need=["patient_id","side","view","full_path","official_split","label"]
missing=[c for c in need if c not in fx.columns]
print("missing columns:", missing if missing else "none")

fx["sp"]=np.where(fx["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
fx["view"]=fx["view"].astype(str).str.upper().str.strip()
fx["side"]=fx["side"].astype(str).str.upper().str.strip()
fx["breast_key"]=fx.patient_id.astype(str)+"_"+fx["side"]
fx["y"]=fx["label"].astype(int) if "label" in fx.columns else \
        fx["pathology"].astype(str).str.upper().str.contains("MALIGNANT").astype(int)

te=fx[fx.sp.eq("test")].copy()
print("\ntest rows %d | breasts %d | views %s"
      % (len(te), te.breast_key.nunique(), dict(te["view"].value_counts())))

ex=str(te["full_path"].iloc[0])
print("\nfull_path example : %s" % ex)
print("exists            : %s" % os.path.exists(ex))
print("files present     : %d / %d" % (te["full_path"].apply(os.path.exists).sum(), len(te)))

rows=[]
for bk,g in te.groupby("breast_key"):
    cc=g.loc[g["view"].str.startswith("CC"),"full_path"].dropna().unique()
    ml=g.loc[g["view"].str.startswith("MLO"),"full_path"].dropna().unique()
    rows.append(dict(breast_key=bk, patient_id=g.patient_id.iloc[0],
                     side=g["side"].iloc[0], label=int(g.y.max()),
                     cc=cc[0] if len(cc) else None, mlo=ml[0] if len(ml) else None,
                     n_cc=len(cc), n_mlo=len(ml)))
P=pd.DataFrame(rows)
both=P[P.cc.notna() & P.mlo.notna()].reset_index(drop=True)

print("\n"+"="*70)
print("  official test breasts total        : %d" % len(P))
print("  with BOTH CC and MLO               : %d   <-- usable" % len(both))
print("  CC only                            : %d" % int((P.cc.notna()&P.mlo.isna()).sum()))
print("  MLO only                           : %d" % int((P.cc.isna()&P.mlo.notna()).sum()))
print("  malignant among usable             : %d (%.1f%%)" % (both.label.sum(),100*both.label.mean()))
print("  both files exist on disk           : %d"
      % int((both.cc.apply(os.path.exists)&both.mlo.apply(os.path.exists)).sum()))
both.to_csv(os.path.join(D,"petrini_pairs_test.csv"),index=False)
print("\n  saved petrini_pairs_test.csv")
print(both.head(3).to_string())

columns: ['patient_id', 'density', 'side', 'view', 'lesion', 'abn_type', 'calc_type', 'calc_dist', 'assessment', 'pathology', 'subtlety', 'full_path', 'cropped image file path', 'mask_path', 'official_split', 'mass_shape', 'mass_margins', 'mask_series', 'full_series', 'img', 'msk', 'source', 'label', 'split']
missing columns: none

test rows 378 | breasts 210 | views {'MLO': np.int64(201), 'CC': np.int64(177)}

full_path example : Mass-Test_P_00016_LEFT_CC/1.3.6.1.4.1.9590.100.1.2.416403281812750683720028031170500130104/1.3.6.1.4.1.9590.100.1.2.245063149211255120613007755642780114172/000000.dcm
exists            : False
files present     : 0 / 378

  official test breasts total        : 210
  with BOTH CC and MLO               : 151   <-- usable
  CC only                            : 19
  MLO only                           : 40
  malignant among usable             : 61 (40.4%)
  both files exist on disk           : 0

  saved petrini_pairs_test.csv
      breast_key patient_id   side  l

In [3]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 23 — LOCATE THE FULL MAMMOGRAM FILES ON DISK
# ══════════════════════════════════════════════════════════════════════════
import os, glob, pandas as pd, numpy as np
D=r"/root/autodl-tmp/CBIS"
ROOTS=["/root/autodl-tmp","/root/autodl-tmp/CBIS","/root/autodl-fs","/root","/root/autodl-tmp/data"]

print("="*78); print("1. LOOKING FOR 'Mass-Test_*' FOLDERS"); print("="*78)
found=[]
for r in ROOTS:
    if not os.path.isdir(r): continue
    for depth in ("*","*/*","*/*/*"):
        hits=glob.glob(os.path.join(r,depth,"Mass-Test_P_*"))
        if hits:
            base=os.path.dirname(hits[0])
            print("  %-45s  %d folders" % (base,len(glob.glob(os.path.join(base,'Mass-Test_P_*')))))
            found.append(base); break
if not found: print("  none found in the searched roots")

print("\n"+"="*78); print("2. WHAT IMAGE FILE TYPES EXIST"); print("="*78)
for ext in ("*.dcm","*.jpg","*.jpeg","*.png"):
    for r in ROOTS:
        if not os.path.isdir(r): continue
        n=len(glob.glob(os.path.join(r,"**",ext),recursive=True))
        if n>50: print("  %-8s %-40s %d files" % (ext,r,n)); break

print("\n"+"="*78); print("3. DOES dicom_info.csv RESOLVE THEM?"); print("="*78)
p=os.path.join(D,"dicom_info.csv")
if os.path.exists(p):
    di=pd.read_csv(p)
    print("  columns: %s" % list(di.columns))
    pc=[c for c in di.columns if "path" in c.lower()]
    print("  path-like columns: %s" % pc)
    for c in pc:
        ex=str(di[c].dropna().iloc[0]); ok=di[c].dropna().astype(str).apply(os.path.exists).sum()
        print("    %-18s exists %5d / %d   example: %s" % (c,ok,di[c].notna().sum(),ex[:90]))
    if "SeriesDescription" in di.columns:
        print("  SeriesDescription counts: %s" % dict(di.SeriesDescription.value_counts()))
else:
    print("  dicom_info.csv not found")

print("\n"+"="*78); print("4. OTHER CSVs THAT MIGHT HOLD RESOLVED PATHS"); print("="*78)
for f in ("cbis_all_exact_matched.csv","cbis_test.csv","full_test_split.csv","paired_test.csv"):
    q=os.path.join(D,f)
    if not os.path.exists(q): continue
    t=pd.read_csv(q); pc=[c for c in t.columns if any(k in c.lower() for k in ("path","jpeg","img","file"))]
    print("\n  %s   rows %d" % (f,len(t)))
    for c in pc:
        s=t[c].dropna().astype(str)
        if not len(s): continue
        ok=s.apply(os.path.exists).sum()
        print("    %-22s exists %5d / %-5d  example: %s" % (c,ok,len(s),s.iloc[0][:80]))

print("\n"+"="*78); print("5. TRY PREPENDING A ROOT TO full_path"); print("="*78)
fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv"))
rel=str(fx["full_path"].dropna().iloc[0])
for r in ROOTS+found:
    if os.path.exists(os.path.join(r,rel)):
        print("  ✅ RESOLVES with root: %s" % r); break
else:
    print("  no simple root prefix works — the DICOMs may not be on this machine,")
    print("  or the full mammograms exist only as converted JPEGs (check section 4)")

1. LOOKING FOR 'Mass-Test_*' FOLDERS
  /root/autodl-tmp/CBIS/cropped_roi_masks_full_from_bbox  378 folders
  /root/autodl-tmp/CBIS/cropped_roi_masks_full_from_bbox  378 folders
  /root/autodl-tmp/CBIS/cropped_roi_masks_full_from_bbox  378 folders

2. WHAT IMAGE FILE TYPES EXIST
  *.dcm    /root/autodl-tmp                         410 files
  *.jpg    /root/autodl-tmp                         10237 files
  *.png    /root/autodl-tmp                         137410 files

3. DOES dicom_info.csv RESOLVE THEM?
  dicom_info.csv not found

4. OTHER CSVs THAT MIGHT HOLD RESOLVED PATHS

  cbis_all_exact_matched.csv   rows 3567
    image_path             exists  3567 / 3567   example: /root/autodl-tmp/CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.2967364033137925996263687801
    pathology              exists     0 / 3567   example: MALIGNANT
    cropped_dicom_path     exists     0 / 3567   example: Mass-Training_P_00001_LEFT_CC_1/1.3.6.1.4.1.9590.100.1.2.10826821301136112420385
    roi_mask_dicom_path    exis

In [6]:
%%bash
cd /root/autodl-tmp
echo "--- ZIP FILES PRESENT ---"
ls -la *.zip

echo; echo "--- UNZIPPING ---"
unzip -o multiple-view-main*.zip

echo; echo "--- EXTRACTED FOLDERS ---"
ls -d multiple-view* 2>/dev/null

cd multiple-view-main 2>/dev/null || cd "$(ls -d multiple-view*/ | head -1)"
echo; echo "--- WORKING DIR: $(pwd) ---"

echo; echo "--- FILES ---"
ls -la

echo; echo "--- PYTHON SCRIPTS ---"
ls *.py 2>/dev/null

echo; echo "--- README ---"
cat README.md 2>/dev/null || cat readme.md 2>/dev/null || echo "no README found"

echo; echo "--- WEIGHT / DOWNLOAD LINKS ---"
grep -rniE "drive\.google|huggingface|hf\.co|download|weight|checkpoint|\.pth|\.pt\b" README.md 2>/dev/null

echo; echo "--- EXISTING WEIGHT FILES ---"
find . \( -name "*.pth" -o -name "*.pt" -o -name "*.ckpt" -o -name "*.bin" \) | head -20

echo; echo "--- REQUIREMENTS ---"
cat requirements.txt 2>/dev/null || echo "no requirements.txt"

--- ZIP FILES PRESENT ---
-rw-r--r-- 1 root root     5684597 Sep  1 10:43 multiple-view-main (1).zip
-rw-r--r-- 1 root root 54411087734 Aug 21 02:51 vindr-mammo-a-large-scale-benchmark-dataset-for-computer-aided-detection-and-diagnosis-in-full-field-digital-mammography-1.0.0.zip

--- UNZIPPING ---
Archive:  multiple-view-main (1).zip
2331b545ebdbd5e8c1da58e13b5dd14ba80d33c2
   creating: multiple-view-main/
   creating: multiple-view-main/Dataloader/
  inflating: multiple-view-main/Dataloader/constants.py  
  inflating: multiple-view-main/Dataloader/dataset_2views.py  
  inflating: multiple-view-main/Dataloader/dataset_full.py  
  inflating: multiple-view-main/Dataloader/example.txt  
  inflating: multiple-view-main/Dataloader/img_process.py  
  inflating: multiple-view-main/LICENSE  
  inflating: multiple-view-main/README.md  
   creating: multiple-view-main/efficientnet_pytorch/
  inflating: multiple-view-main/efficientnet_pytorch/__init__.py  
   creating: multiple-view-main/efficien

In [7]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 24 — VERIFY paired_test.csv IS THE OFFICIAL TEST SET
# ══════════════════════════════════════════════════════════════════════════
import os, numpy as np, pandas as pd
D=r"/root/autodl-tmp/CBIS"

pt=pd.read_csv(os.path.join(D,"paired_test.csv"))
pp=pd.read_csv(os.path.join(D,"petrini_pairs_test.csv"))     # from CELL 22
print("paired_test.csv  rows %d  columns %s" % (len(pt),list(pt.columns)))
print("petrini_pairs    rows %d" % len(pp))

a=set(pt.breast_key.astype(str)); b=set(pp.breast_key.astype(str))
print("\n  breast_keys in both        : %d" % len(a&b))
print("  only in paired_test.csv    : %d" % len(a-b))
print("  only in petrini_pairs      : %d" % len(b-a))

fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv"))
fx["sp"]=np.where(fx["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
fx["breast_key"]=fx.patient_id.astype(str)+"_"+fx["side"].astype(str).str.upper().str.strip()
off_te=set(fx.loc[fx.sp.eq("test"),"breast_key"]); off_tr=set(fx.loc[fx.sp.eq("train"),"breast_key"])
print("\n  paired breasts in OFFICIAL TEST  : %d / %d" % (len(a&off_te),len(a)))
print("  paired breasts in OFFICIAL TRAIN : %d   <-- must be 0" % len(a&off_tr))

m=pt.merge(pp[["breast_key","label"]],on="breast_key",how="inner",suffixes=("_pt","_pp"))
if "label_pt" in m.columns and "label_pp" in m.columns:
    print("  labels agree between the two files: %d / %d" % ((m.label_pt==m.label_pp).sum(),len(m)))

print("\n  cc_jpeg  exist : %d / %d" % (pt.cc_jpeg.astype(str).apply(os.path.exists).sum(),len(pt)))
print("  mlo_jpeg exist : %d / %d" % (pt.mlo_jpeg.astype(str).apply(os.path.exists).sum(),len(pt)))
print("  malignant      : %d (%.1f%%)" % (pt.label.sum(),100*pt.label.mean()))
print("\n  example cc : %s" % pt.cc_jpeg.iloc[0])
print("  example mlo: %s" % pt.mlo_jpeg.iloc[0])

paired_test.csv  rows 151  columns ['breast_key', 'patient_id', 'side', 'cc_jpeg', 'mlo_jpeg', 'label', 'label_name', 'abn_type', 'shape_feat', 'margin_feat', 'calctype_feat', 'calcdist_feat', 'breast_density', 'subtlety', 'assessment']
petrini_pairs    rows 151

  breast_keys in both        : 0
  only in paired_test.csv    : 151
  only in petrini_pairs      : 151

  paired breasts in OFFICIAL TEST  : 0 / 151
  paired breasts in OFFICIAL TRAIN : 0   <-- must be 0
  labels agree between the two files: 0 / 0

  cc_jpeg  exist : 151 / 151
  mlo_jpeg exist : 151 / 151
  malignant      : 61 (40.4%)

  example cc : /root/autodl-tmp/CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.30820586311062570442302321942433426184/2-272.jpg
  example mlo: /root/autodl-tmp/CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.381440141511137044327302306604206077287/2-273.jpg


In [8]:
%%bash
echo "--- WHERE DID IT GO ---"
find /root -maxdepth 5 -type d -name "multiple-view*" 2>/dev/null
find /root -maxdepth 5 -name "multi_view_clf_test.py" 2>/dev/null

DIR=$(dirname $(find /root -maxdepth 5 -name "multi_view_clf_test.py" 2>/dev/null | head -1))
echo; echo "--- USING: $DIR ---"
cd "$DIR" || exit 1
ls -la

echo; echo "--- SAMPLES ---"
ls -la samples/ | head

echo; echo "--- SAMPLE IMAGE SIZES (this tells us what resolution to resize yours to) ---"
python3 - <<'PY'
import glob, cv2
for f in sorted(glob.glob("samples/*"))[:6]:
    im = cv2.imread(f, cv2.IMREAD_UNCHANGED)
    print("  %-55s %s  dtype %s" % (f, None if im is None else im.shape, None if im is None else im.dtype))
PY

echo; echo "--- IS THERE A models FOLDER ---"
ls -la models/ 2>/dev/null || echo "  no models folder yet — must be created and filled"

--- WHERE DID IT GO ---
/root/autodl-tmp/multiple-view-main
/root/autodl-tmp/multiple-view-main/multi_view_clf_test.py

--- USING: /root/autodl-tmp/multiple-view-main ---
total 44
drwxr-xr-x  5 root root 4096 Aug 30 04:45 .
drwxr-xr-x 13 root root 4096 Sep  1 10:48 ..
drwxr-xr-x  2 root root  139 Aug 30 04:45 Dataloader
-rw-r--r--  1 root root 1067 Aug 30 04:45 LICENSE
-rw-r--r--  1 root root 2970 Aug 30 04:45 README.md
drwxr-xr-x  3 root root   96 Aug 30 04:45 efficientnet_pytorch
-rwxr-xr-x  1 root root 6968 Aug 30 04:45 multi_view_clf_test.py
drwxr-xr-x  2 root root 4096 Aug 30 04:45 samples
-rwxr-xr-x  1 root root 6087 Aug 30 04:45 single_full_net_2024.py
-rw-r--r--  1 root root 7621 Aug 30 04:45 two_views_net_2024.py

--- SAMPLES ---
total 5512
drwxr-xr-x 2 root root    4096 Aug 30 04:45 .
drwxr-xr-x 5 root root    4096 Aug 30 04:45 ..
-rw-r--r-- 1 root root  821626 Aug 30 04:45 Calc-Test_P_00041_LEFT_CC.png
-rw-r--r-- 1 root root  824416 Aug 30 04:45 Calc-Test_P_00041_LEFT_MLO.pn

In [9]:
%%bash
pip install -q timm opencv-python-headless huggingface_hub
python3 -c "import torch, timm, cv2, numpy; print('torch', torch.__version__, '| timm', timm.__version__, '| cuda', torch.cuda.is_available())"


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


torch 2.8.0+cu128 | timm 1.0.27 | cuda True


In [14]:
%%bash
cd /root/autodl-tmp/multiple-view-main
grep -n "models\|\.pth\|\.pt'\|\.pt\"\|torch.load\|imread\|resize\|65535\|/255\|IMREAD" multi_view_clf_test.py

50:                model_file = 'models/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt'
52:                model_file = 'models/2024-05-14-18h55m_50ep_best_model_AUC_08548_Best_CBIS_EFB3_CVUD.pt'
54:            model_file = 'models/2024-08-06-18h01m_50ep_best_model_AUC_08187_CVUBP_VINDR_CONVNEXT.pt'
65:                self.model_file = 'models/2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt'
67:                self.model_file = 'models/2024-07-01-12h34m_50ep_best_model_AUC_08856_BEST_2024_Ef-B3_CBIS.pt'
69:            self.model_file = 'models/2024-08-07-18h35m_50ep_best_model_AUC_08655_ConvNext_VinDr_Collab.pt'
74:        self.model.load_state_dict(torch.load(self.model_file, map_location=self.device))
94:        image /= 65535
96:        image /= 65535
182:    # First assemble 2-views network with the single view models
189:    image = cv2.imread(file_cc, cv2.IMREAD_UNCHANGED)
196:    image = cv2.imread(file_mlo, cv2.IMREAD_UNCHANGED)


In [18]:
%%bash
cd /root/autodl-tmp/multiple-view-main
echo "############ multi_view_clf_test.py ############"
cat multi_view_clf_test.py
echo; echo "############ two_views_net_2024.py ############"
cat two_views_net_2024.py

############ multi_view_clf_test.py ############
# 2 views CLASSIFIER Improved - test script
#
# Author: Daniel Petrini
#
# Test inference for 2 views mammograms with 2024 nwtworks
#
# run: python3 2views_clf_test.py -c [cc image file] -m [mlo image file]
#
# DGPP 12/Dec/2024


import argparse
import numpy as np
import torch
from torch.autograd import Variable
import cv2

from two_views_net_2024 import SideMIDBreastModel

# ****************************************** New

EXPERIMENT_TYPE = 'PBC'         #   'DC', 'PBC'

# ******************************************

DEVICE = 'gpu'
gpu_number = 0

# Configs for single model assembling
TOP_MODE = 'TOP_EF_NET' 
TOP_LAYER_N_BLOCKS = 2
STRIDES = 2
TOP_LAYER_BLOCK_TYPE = 'mbconv'
USE_AVG_POOL = True
TRAIN_DS_MEAN = 13369   # Mean of all files in training 


class LoadModel():
    def __init__(self, device, network) -> None:
        self.device = device
        self.network = network

    def get_single_model_file(self):
        """
        Loa

In [17]:
%%bash
source /etc/network_turbo
mkdir -p /root/autodl-tmp/multiple-view-main/models
hf download dpetrini/cbis-ddsm-efficientnetb3-multiple-view \
  --local-dir /root/autodl-tmp/multiple-view-main/models
ls -la /root/autodl-tmp/multiple-view-main/models

设置成功!
注意：
1. 仅限学术用途和加速访问github/huggingface，不承诺稳定性
2. 开启加速后对访问其他资源如pip源等会*更慢*


Fetching 3 files: 100%|██████████| 3/3 [00:29<00:00,  9.94s/it]


✓ Downloaded
  path: /root/autodl-tmp/multiple-view-main/models
total 504848
drwxr-xr-x 3 root root       162 Sep  1 11:14 .
drwxr-xr-x 6 root root      4096 Sep  1 10:57 ..
drwxr-xr-x 3 root root        33 Sep  1 11:14 .cache
-rw-r--r-- 1 root root      1519 Sep  1 11:14 .gitattributes
-rw-r--r-- 1 root root 516949042 Sep  1 11:14 2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt
-rw-r--r-- 1 root root       563 Sep  1 11:14 README.md


In [19]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 25 — CONVERT ONE IMAGE AND CHECK IT MATCHES THEIR EXPECTED STATISTICS
# ══════════════════════════════════════════════════════════════════════════
import os, glob, cv2, numpy as np, pandas as pd
D=r"/root/autodl-tmp/CBIS"; REPO=r"/root/autodl-tmp/multiple-view-main"
TRAIN_DS_MEAN=13369; H,W=1152,896

print("="*74); print("THEIR SAMPLE IMAGES"); print("="*74)
for f in sorted(glob.glob(os.path.join(REPO,"samples","*.png")))[:3]:
    im=cv2.imread(f,cv2.IMREAD_UNCHANGED)
    print("  %-42s %s %s  min %5d  max %5d  mean %8.1f"
          % (os.path.basename(f),im.shape,im.dtype,im.min(),im.max(),im.mean()))
print("\n  script subtracts TRAIN_DS_MEAN = %d then divides by 65535" % TRAIN_DS_MEAN)

print("\n"+"="*74); print("YOUR IMAGE, CONVERTED"); print("="*74)
pt=pd.read_csv(os.path.join(D,"paired_test.csv"))
src=str(pt.cc_jpeg.iloc[0]); print("  source: %s" % src)
raw=cv2.imread(src,cv2.IMREAD_UNCHANGED)
print("  as loaded    %s %s  min %d  max %d  mean %.1f" % (raw.shape,raw.dtype,raw.min(),raw.max(),raw.mean()))
if raw.ndim==3: raw=cv2.cvtColor(raw,cv2.COLOR_BGR2GRAY)
rs=cv2.resize(raw,(W,H),interpolation=cv2.INTER_AREA)
conv=(rs.astype(np.float32)*257.0).clip(0,65535).astype(np.uint16)
print("  converted    %s %s  min %d  max %d  mean %.1f"
      % (conv.shape,conv.dtype,conv.min(),conv.max(),conv.mean()))
print("\n  their sample mean vs yours: compare the two 'mean' figures above.")
print("  if yours is wildly different from theirs, the 8-bit JPEG was windowed")
print("  differently and a plain x257 scale is not faithful.")

os.makedirs(os.path.join(D,"petrini_input"),exist_ok=True)
cv2.imwrite(os.path.join(D,"petrini_input","_test_cc.png"),conv)
print("\n  wrote %s" % os.path.join(D,"petrini_input","_test_cc.png"))

THEIR SAMPLE IMAGES
  Calc-Test_P_00041_LEFT_CC.png              (1152, 896) uint16  min     0  max 65535  mean  10847.4
  Calc-Test_P_00041_LEFT_MLO.png             (1152, 896) uint16  min     0  max 65535  mean  12357.8
  Calc-Test_P_00127_RIGHT_CC.png             (1152, 896) uint16  min     0  max 65535  mean  11576.4

  script subtracts TRAIN_DS_MEAN = 13369 then divides by 65535

YOUR IMAGE, CONVERTED
  source: /root/autodl-tmp/CBIS/jpeg/1.3.6.1.4.1.9590.100.1.2.30820586311062570442302321942433426184/2-272.jpg
  as loaded    (384, 385) uint8  min 0  max 255  mean 145.1
  converted    (1152, 896) uint16  min 0  max 65535  mean 37295.5

  their sample mean vs yours: compare the two 'mean' figures above.
  if yours is wildly different from theirs, the 8-bit JPEG was windowed
  differently and a plain x257 scale is not faithful.

  wrote /root/autodl-tmp/CBIS/petrini_input/_test_cc.png


In [20]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 26 — FIND THE REAL FULL MAMMOGRAMS AND REBUILD CC/MLO PAIRS
# ══════════════════════════════════════════════════════════════════════════
import os, re, cv2, numpy as np, pandas as pd
D=r"/root/autodl-tmp/CBIS"; REPO=r"/root/autodl-tmp/multiple-view-main"
H,W=1152,896

print("="*78); print("1. full_test_split.csv"); print("="*78)
ft=pd.read_csv(os.path.join(D,"full_test_split.csv"))
print("  rows %d  columns %s" % (len(ft),list(ft.columns)))
print("\n  first 3 full_case_id values:")
for v in ft["full_case_id"].head(3): print("    %s" % v)

print("\n"+"="*78); print("2. ARE THESE ACTUALLY FULL MAMMOGRAMS?"); print("="*78)
for i in range(min(5,len(ft))):
    p=str(ft["full_jpeg_path"].iloc[i])
    im=cv2.imread(p,cv2.IMREAD_UNCHANGED)
    print("  %-38s %s  mean %.1f" % (ft['full_case_id'].iloc[i][:38],
          None if im is None else im.shape, -1 if im is None else im.mean()))
print("\n  full mammograms should be roughly 3000x5000 and tall, not small squares")

print("\n"+"="*78); print("3. PARSE SIDE AND VIEW, BUILD PAIRS"); print("="*78)
def parse(cid):
    s=str(cid).upper()
    side="LEFT" if "_LEFT" in s else ("RIGHT" if "_RIGHT" in s else None)
    view="MLO" if re.search(r"_MLO(\b|_)",s) else ("CC" if re.search(r"_CC(\b|_)",s) else None)
    m=re.search(r"(P_\d+)",s)
    return (m.group(1) if m else None), side, view
ft[["pid","side","view"]]=ft["full_case_id"].apply(lambda c: pd.Series(parse(c)))
ft["breast_key"]=ft.pid.astype(str)+"_"+ft.side.astype(str)
print("  parsed views: %s" % dict(ft["view"].value_counts(dropna=False)))
print("  parsed sides: %s" % dict(ft["side"].value_counts(dropna=False)))
print("  breasts: %d" % ft.breast_key.nunique())

fx=pd.read_csv(os.path.join(D,"cbis_mass_fixed.csv"))
fx["sp"]=np.where(fx["official_split"].astype(str).str.lower().str.contains("test"),"test","train")
fx["breast_key"]=fx.patient_id.astype(str)+"_"+fx["side"].astype(str).str.upper().str.strip()
fx["y"]=fx["label"].astype(int)
lab=fx[fx.sp.eq("test")].groupby("breast_key").y.max()
off_te=set(lab.index); off_tr=set(fx.loc[fx.sp.eq("train"),"breast_key"])

rows=[]
for bk,g in ft.groupby("breast_key"):
    cc=g.loc[g["view"].eq("CC"),"full_jpeg_path"].dropna().unique()
    ml=g.loc[g["view"].eq("MLO"),"full_jpeg_path"].dropna().unique()
    if len(cc) and len(ml) and bk in off_te:
        rows.append(dict(breast_key=bk,label=int(lab[bk]),cc=cc[0],mlo=ml[0]))
P=pd.DataFrame(rows)
print("\n  breasts with BOTH views AND in official test : %d" % len(P))
print("  any in official TRAIN (must be 0)            : %d"
      % (0 if len(P)==0 else sum(P.breast_key.isin(off_tr))))
if len(P):
    print("  malignant %d (%.1f%%)" % (P.label.sum(),100*P.label.mean()))
    print("  both files exist: %d" % int((P.cc.apply(os.path.exists)&P.mlo.apply(os.path.exists)).sum()))
    P.to_csv(os.path.join(D,"petrini_fullmammo_pairs.csv"),index=False)
    print("  saved petrini_fullmammo_pairs.csv")

print("\n"+"="*78); print("4. CONVERSION CHECK ON A REAL FULL MAMMOGRAM"); print("="*78)
if len(P):
    raw=cv2.imread(str(P.cc.iloc[0]),cv2.IMREAD_UNCHANGED)
    if raw is not None:
        if raw.ndim==3: raw=cv2.cvtColor(raw,cv2.COLOR_BGR2GRAY)
        print("  original  %s %s  mean %.1f" % (raw.shape,raw.dtype,raw.mean()))
        rs=cv2.resize(raw,(W,H),interpolation=cv2.INTER_AREA)
        conv=(rs.astype(np.float32)*257.0).clip(0,65535).astype(np.uint16)
        print("  converted %s %s  mean %.1f" % (conv.shape,conv.dtype,conv.mean()))
        print("  THEIR samples mean ~10847-12358   <-- yours should be close to this")
        os.makedirs(os.path.join(D,"petrini_input"),exist_ok=True)
        cv2.imwrite(os.path.join(D,"petrini_input","_check_cc.png"),conv)
        print("  wrote petrini_input/_check_cc.png")

1. full_test_split.csv
  rows 362  columns ['full_case_id', 'patient_id', 'full_jpeg_path', 'roi_mask_paths', 'n_rois', 'label_name', 'abn_type', 'split', 'breast_density', 'subtlety', 'assessment']

  first 3 full_case_id values:
    Calc-Test_P_00485_LEFT_CC
    Mass-Test_P_00016_LEFT_CC
    Mass-Test_P_00016_LEFT_MLO

2. ARE THESE ACTUALLY FULL MAMMOGRAMS?
  Calc-Test_P_00485_LEFT_CC              (6032, 4056)  mean 85.1
  Mass-Test_P_00016_LEFT_CC              (4006, 1846)  mean 35.0
  Mass-Test_P_00016_LEFT_MLO             (5491, 2011)  mean 68.1
  Mass-Test_P_00017_LEFT_CC              (5904, 3200)  mean 50.6
  Mass-Test_P_00017_LEFT_MLO             (5952, 3352)  mean 61.6

  full mammograms should be roughly 3000x5000 and tall, not small squares

3. PARSE SIDE AND VIEW, BUILD PAIRS
  parsed views: {'MLO': np.int64(191), 'CC': np.int64(171)}
  parsed sides: {'RIGHT': np.int64(182), 'LEFT': np.int64(180)}
  breasts: 211

  breasts with BOTH views AND in official test : 151
  any in

In [21]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 27 — CONVERT ALL 151 CC/MLO PAIRS TO PETRINI'S INPUT FORMAT
#   1152 x 896, uint16 PNG, as their sample files are.
#   Writes to CBIS/petrini_input/ and saves a manifest.
# ══════════════════════════════════════════════════════════════════════════
import os, cv2, numpy as np, pandas as pd, time
D=r"/root/autodl-tmp/CBIS"; OUT=os.path.join(D,"petrini_input")
H,W=1152,896
os.makedirs(OUT,exist_ok=True)

P=pd.read_csv(os.path.join(D,"petrini_fullmammo_pairs.csv"))
print("pairs to convert: %d" % len(P))

def conv(src,dst):
    im=cv2.imread(str(src),cv2.IMREAD_UNCHANGED)
    if im is None: return None
    if im.ndim==3: im=cv2.cvtColor(im,cv2.COLOR_BGR2GRAY)
    if im.dtype==np.uint8:
        im=cv2.resize(im,(W,H),interpolation=cv2.INTER_AREA)
        im=(im.astype(np.float32)*257.0).clip(0,65535).astype(np.uint16)
    else:
        im=cv2.resize(im,(W,H),interpolation=cv2.INTER_AREA).astype(np.uint16)
    cv2.imwrite(dst,im)
    return float(im.mean())

rows=[]; t0=time.time(); means=[]
for i,r in P.iterrows():
    bk=str(r["breast_key"])
    dcc=os.path.join(OUT,bk+"_CC.png"); dml=os.path.join(OUT,bk+"_MLO.png")
    m1=conv(r["cc"],dcc); m2=conv(r["mlo"],dml)
    if m1 is None or m2 is None:
        print("  FAILED: %s" % bk); continue
    means+= [m1,m2]
    rows.append(dict(breast_key=bk,label=int(r["label"]),cc_png=dcc,mlo_png=dml))
    if (i+1)%25==0: print("  %3d/%d  (%.0fs)" % (i+1,len(P),time.time()-t0))

M=pd.DataFrame(rows)
M.to_csv(os.path.join(D,"petrini_input_manifest.csv"),index=False)

print("\n"+"="*74)
print("  converted %d breasts (%d images) in %.0fs" % (len(M),2*len(M),time.time()-t0))
print("  malignant %d (%.1f%%)" % (M.label.sum(),100*M.label.mean()))
print("  converted mean  : %.1f  (min %.1f, max %.1f)"
      % (np.mean(means),np.min(means),np.max(means)))
print("  their samples   : ~10847-12358")
print("  all files exist : %d / %d"
      % (int((M.cc_png.apply(os.path.exists)&M.mlo_png.apply(os.path.exists)).sum()),len(M)))
print("  saved petrini_input_manifest.csv")
print("  images in %s" % OUT)
print("="*74)

pairs to convert: 151
   25/151  (3s)
   50/151  (6s)
   75/151  (9s)
  100/151  (12s)
  125/151  (15s)
  150/151  (19s)

  converted 151 breasts (302 images) in 19s
  malignant 61 (40.4%)
  converted mean  : 13958.8  (min 3677.1, max 29127.0)
  their samples   : ~10847-12358
  all files exist : 151 / 151
  saved petrini_input_manifest.csv
  images in /root/autodl-tmp/CBIS/petrini_input


In [23]:
%%bash
export HF_ENDPOINT=https://hf-mirror.com
mkdir -p /root/autodl-tmp/multiple-view-main/models
hf download dpetrini/cbis-ddsm-efficientnetb3-multiple-view \
  --local-dir /root/autodl-tmp/multiple-view-main/models
echo; echo "--- WHAT ARRIVED ---"
ls -la /root/autodl-tmp/multiple-view-main/models

Fetching 3 files: 100%|██████████| 3/3 [00:00<00:00, 2352.39it/s]


✓ Downloaded
  path: /root/autodl-tmp/multiple-view-main/models

--- WHAT ARRIVED ---
total 504848
drwxr-xr-x 3 root root       162 Sep  1 11:14 .
drwxr-xr-x 7 root root      4096 Sep  1 11:20 ..
drwxr-xr-x 3 root root        33 Sep  1 11:14 .cache
-rw-r--r-- 1 root root      1519 Sep  1 11:14 .gitattributes
-rw-r--r-- 1 root root 516949042 Sep  1 11:14 2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt
-rw-r--r-- 1 root root       563 Sep  1 11:14 README.md


In [24]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 28 — CHECK THE WEIGHT FILES ARE PRESENT AND CORRECTLY NAMED
# ══════════════════════════════════════════════════════════════════════════
import os, glob, shutil
M=r"/root/autodl-tmp/multiple-view-main/models"
NEED=["2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt",
      "2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt"]

print("="*74); print("FILES CURRENTLY IN models/"); print("="*74)
have=sorted(glob.glob(os.path.join(M,"**","*"),recursive=True))
if not have: print("  (empty)")
for f in have:
    if os.path.isfile(f):
        print("  %-72s %.1f MB" % (os.path.relpath(f,M), os.path.getsize(f)/1e6))

print("\n"+"="*74); print("WHAT THE SCRIPT NEEDS"); print("="*74)
for n in NEED:
    p=os.path.join(M,n)
    print("  %-72s %s" % (n[:72], "OK" if os.path.exists(p) else "MISSING"))

print("\n"+"="*74); print("AUTO-MATCH BY KEYWORD"); print("="*74)
pts=[f for f in have if f.endswith(".pt") and os.path.isfile(f)]
for n in NEED:
    if os.path.exists(os.path.join(M,n)): continue
    key = "08393" if "08393" in n else "08643"
    cand=[f for f in pts if key in os.path.basename(f)]
    if cand:
        shutil.copy(cand[0], os.path.join(M,n))
        print("  copied %s\n      -> %s" % (os.path.basename(cand[0]), n))
    else:
        print("  no file containing '%s' found — download it manually" % key)

print("\nfinal check:")
for n in NEED:
    print("  %s  %s" % ("OK     " if os.path.exists(os.path.join(M,n)) else "MISSING", n[:70]))

FILES CURRENTLY IN models/
  2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt 516.9 MB
  README.md                                                                0.0 MB

WHAT THE SCRIPT NEEDS
  2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt      MISSING
  2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP OK

AUTO-MATCH BY KEYWORD
  no file containing '08393' found — download it manually

final check:
  MISSING  2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt
  OK       2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVU


In [25]:
%%bash
cd /root/autodl-tmp
find /root -name "*2024-05-13*" 2>/dev/null
unzip -o /root/autodl-tmp/*2024-05-13*.zip -d /root/autodl-tmp/multiple-view-main/models/
ls -la /root/autodl-tmp/multiple-view-main/models/

/root/autodl-tmp/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt


unzip:  cannot find or open /root/autodl-tmp/*2024-05-13*.zip, /root/autodl-tmp/*2024-05-13*.zip.zip or /root/autodl-tmp/*2024-05-13*.zip.ZIP.

No zipfiles found.


total 504848
drwxr-xr-x 3 root root       162 Sep  1 11:14 .
drwxr-xr-x 7 root root      4096 Sep  1 11:20 ..
drwxr-xr-x 3 root root        33 Sep  1 11:14 .cache
-rw-r--r-- 1 root root      1519 Sep  1 11:14 .gitattributes
-rw-r--r-- 1 root root 516949042 Sep  1 11:14 2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt
-rw-r--r-- 1 root root       563 Sep  1 11:14 README.md


In [27]:
%%bash
REPO=/root/autodl-tmp/multiple-view-main
NEED="2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt"

echo "--- SEARCHING FOR THE FILE ---"
find /root -name "*08393*" -type f 2>/dev/null

SRC=$(find /root -name "*08393*.pt" -type f 2>/dev/null | head -1)
if [ -z "$SRC" ]; then
  echo "NOT FOUND. Is it still zipped? Looking for zips..."
  find /root -name "*08393*" 2>/dev/null
  exit 1
fi
echo; echo "found: $SRC"

mkdir -p "$REPO/models"
if [ "$SRC" != "$REPO/models/$NEED" ]; then
  cp "$SRC" "$REPO/models/$NEED"
  echo "copied into $REPO/models/$NEED"
fi

echo; echo "--- models/ CONTENTS ---"
ls -la "$REPO/models"/*.pt

echo; echo "--- BOTH FILES PRESENT? ---"
for f in "2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt" \
         "2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt"; do
  [ -f "$REPO/models/$f" ] && echo "  OK      $f" || echo "  MISSING $f"
done

echo; echo "=============== SANITY CHECK — MUST PRINT 0.1214 ==============="
cd "$REPO"
python3 multi_view_clf_test.py \
  -c samples/Calc-Test_P_00127_RIGHT_CC.png \
  -m samples/Calc-Test_P_00127_RIGHT_MLO.png

--- SEARCHING FOR THE FILE ---
/root/autodl-tmp/multiple-view-main/models/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt
/root/autodl-tmp/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt

found: /root/autodl-tmp/multiple-view-main/models/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt

--- models/ CONTENTS ---
-rw-r--r-- 1 root root 138274966 Sep  1 11:31 /root/autodl-tmp/multiple-view-main/models/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt
-rw-r--r-- 1 root root 516949042 Sep  1 11:14 /root/autodl-tmp/multiple-view-main/models/2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt

--- BOTH FILES PRESENT? ---
  OK      2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt
  OK      2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt

=============== SANITY CHECK — MUST PRINT 0.1214 ===============
CC image path: samples/Calc-Test_P_00127_RIGHT_

Traceback (most recent call last):
  File "/root/autodl-tmp/multiple-view-main/multi_view_clf_test.py", line 210, in <module>
    main()
  File "/root/autodl-tmp/multiple-view-main/multi_view_clf_test.py", line 184, in main
    model, device = MyModel.load_model(dataset)
                    ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/autodl-tmp/multiple-view-main/multi_view_clf_test.py", line 80, in load_model
    model = SideMIDBreastModel(self.device, model_file, self.network, TOP_LAYER_N_BLOCKS,
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/autodl-tmp/multiple-view-main/two_views_net_2024.py", line 70, in __init__
    self.two_views_clf = TwoViewsMIDBreastClassifier(device, model_file, network, exp_type, dataset)
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/root/autodl-tmp/multiple-view-main/two_views_net_2024.py", line 28, in __init__
    self.single_clf_full.loa

CalledProcessError: Command 'b'REPO=/root/autodl-tmp/multiple-view-main\nNEED="2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt"\n\necho "--- SEARCHING FOR THE FILE ---"\nfind /root -name "*08393*" -type f 2>/dev/null\n\nSRC=$(find /root -name "*08393*.pt" -type f 2>/dev/null | head -1)\nif [ -z "$SRC" ]; then\n  echo "NOT FOUND. Is it still zipped? Looking for zips..."\n  find /root -name "*08393*" 2>/dev/null\n  exit 1\nfi\necho; echo "found: $SRC"\n\nmkdir -p "$REPO/models"\nif [ "$SRC" != "$REPO/models/$NEED" ]; then\n  cp "$SRC" "$REPO/models/$NEED"\n  echo "copied into $REPO/models/$NEED"\nfi\n\necho; echo "--- models/ CONTENTS ---"\nls -la "$REPO/models"/*.pt\n\necho; echo "--- BOTH FILES PRESENT? ---"\nfor f in "2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt" \\\n         "2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt"; do\n  [ -f "$REPO/models/$f" ] && echo "  OK      $f" || echo "  MISSING $f"\ndone\n\necho

In [28]:
%%bash
echo "================= DISK SPACE ================="
df -h /root/autodl-tmp | tail -2

echo; echo "================= THE TWO COPIES ================="
ls -la /root/autodl-tmp/*08393*.pt 2>/dev/null
ls -la /root/autodl-tmp/multiple-view-main/models/*08393*.pt 2>/dev/null

echo; echo "================= FREE THE EASY SPACE ================="
rm -rf ~/.cache/pip ~/.cache/torch 2>/dev/null
rm -rf /root/autodl-tmp/multiple-view-main/models/.cache 2>/dev/null
find /root/autodl-tmp -name "__pycache__" -type d -exec rm -rf {} + 2>/dev/null
conda clean -a -y >/dev/null 2>&1
echo "after cache cleanup:"; df -h /root/autodl-tmp | tail -1

================= DISK SPACE =================
Filesystem      Size  Used Avail Use% Mounted on
/dev/nvme1n1     84G   34G   51G  41% /root/autodl-tmp

================= THE TWO COPIES =================
-rw-r--r-- 1 root root 138274966 Sep  1 11:34 /root/autodl-tmp/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt
-rw-r--r-- 1 root root 138274966 Sep  1 11:31 /root/autodl-tmp/multiple-view-main/models/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt

================= FREE THE EASY SPACE =================
after cache cleanup:
/dev/nvme1n1     84G   34G   51G  41% /root/autodl-tmp


In [29]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 30 — WHICH COPY OF THE WEIGHT FILE IS INTACT?
# ══════════════════════════════════════════════════════════════════════════
import os, torch, shutil
A="/root/autodl-tmp/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt"
B="/root/autodl-tmp/multiple-view-main/models/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt"

for tag,p in (("original (autodl-tmp)",A),("copy (models/)",B)):
    if not os.path.exists(p):
        print("  %-24s MISSING" % tag); continue
    sz=os.path.getsize(p)/1e6
    try:
        sd=torch.load(p,map_location="cpu")
        print("  %-24s %7.1f MB   VALID   %d tensors" % (tag,sz,len(sd) if hasattr(sd,'__len__') else -1))
    except Exception as e:
        print("  %-24s %7.1f MB   CORRUPT  %s" % (tag,sz,str(e)[:60]))

  original (autodl-tmp)      138.3 MB   CORRUPT  PytorchStreamReader failed reading zip archive: failed findi
  copy (models/)             138.3 MB   CORRUPT  PytorchStreamReader failed reading zip archive: failed findi


In [30]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 31 — DROP THE CORRUPT FILE AND TEST WITHOUT IT
# ══════════════════════════════════════════════════════════════════════════
import os, torch, textwrap, subprocess
REPO="/root/autodl-tmp/multiple-view-main"; M=os.path.join(REPO,"models")
BIG=os.path.join(M,"2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt")

print("="*70); print("1. IS THE TWO-VIEWS CHECKPOINT VALID?"); print("="*70)
try:
    sd=torch.load(BIG,map_location="cpu")
    print("  VALID — %d tensors, %.1f MB" % (len(sd),os.path.getsize(BIG)/1e6))
    ks=list(sd.keys())
    print("  first keys: %s" % ks[:3])
    print("  has feature-extractor weights: %s"
          % any("single_clf_core" in k or "two_views_clf" in k for k in ks))
except Exception as e:
    print("  CORRUPT TOO: %s" % e); raise SystemExit

print("\n"+"="*70); print("2. REMOVE THE CORRUPT FILES"); print("="*70)
for p in ["/root/autodl-tmp/2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt",
          os.path.join(M,"2024-05-13-21h48m_50ep_best_model_AUC_08393_Best_CBIS_EFB3-CVUBP.pt")]:
    if os.path.exists(p): os.remove(p); print("  deleted %s" % p)

print("\n"+"="*70); print("3. WRITE THE PATCHED RUNNER"); print("="*70)
patch = textwrap.dedent('''
    import os, torch
    _rl = torch.load
    _rlsd = torch.nn.Module.load_state_dict
    def _load(f,*a,**k):
        if isinstance(f,(str,os.PathLike)) and not os.path.exists(str(f)):
            print("  [patch] skipping missing init file:", os.path.basename(str(f)))
            return {}
        return _rl(f,*a,**k)
    def _lsd(self,sd,strict=True,**k):
        if isinstance(sd,dict) and len(sd)==0: return None
        return _rlsd(self,sd,strict=strict,**k)
    torch.load = _load
    torch.nn.Module.load_state_dict = _lsd
    import multi_view_clf_test as M
    M.main()
''')
open(os.path.join(REPO,"mv_patched.py"),"w").write(patch)
print("  wrote mv_patched.py")

print("\n"+"="*70); print("4. SANITY CHECK — MUST PRINT 0.1214"); print("="*70)
r=subprocess.run(["python3","mv_patched.py",
                  "-c","samples/Calc-Test_P_00127_RIGHT_CC.png",
                  "-m","samples/Calc-Test_P_00127_RIGHT_MLO.png"],
                 cwd=REPO,capture_output=True,text=True)
print(r.stdout)
if r.stderr: print("STDERR:\n", r.stderr[-1500:])

1. IS THE TWO-VIEWS CHECKPOINT VALID?
  CORRUPT TOO: PytorchStreamReader failed reading zip archive: failed finding central directory


SystemExit: 

/root/miniconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [31]:
%%bash
export HF_ENDPOINT=https://hf-mirror.com
REPO=/root/autodl-tmp/multiple-view-main
rm -f "$REPO/models"/*.pt
rm -rf "$REPO/models/.cache"
mkdir -p "$REPO/models"

hf download dpetrini/cbis-ddsm-efficientnetb3-multiple-view --local-dir "$REPO/models"

echo; echo "--- WHAT ARRIVED (following links) ---"
ls -laL "$REPO/models"
echo; echo "--- REAL SIZE ON DISK ---"
du -sh "$REPO/models"
df -h /root/autodl-tmp | tail -1

Fetching 3 files: 100%|██████████| 3/3 [26:49<00:00, 536.66s/it]


✓ Downloaded
  path: /root/autodl-tmp/multiple-view-main/models

--- WHAT ARRIVED (following links) ---
total 504848
drwxr-xr-x 3 root root       162 Sep  1 12:10 .
drwxr-xr-x 6 root root      4096 Sep  1 11:34 ..
drwxr-xr-x 3 root root        33 Sep  1 11:43 .cache
-rw-r--r-- 1 root root      1519 Sep  1 11:43 .gitattributes
-rw-r--r-- 1 root root 516949042 Sep  1 12:10 2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt
-rw-r--r-- 1 root root       563 Sep  1 11:43 README.md

--- REAL SIZE ON DISK ---
494M	/root/autodl-tmp/multiple-view-main/models
/dev/nvme1n1     84G   34G   51G  40% /root/autodl-tmp


In [32]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 32 — VERIFY THE CHECKPOINT, THEN RUN WITHOUT THE SINGLE-VIEW FILE
# ══════════════════════════════════════════════════════════════════════════
import os, torch, textwrap, subprocess
REPO="/root/autodl-tmp/multiple-view-main"; M=os.path.join(REPO,"models")
BIG=os.path.join(M,"2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt")

print("="*70); print("1. CHECKPOINT INTEGRITY"); print("="*70)
print("  size   : %.1f MB" % (os.path.getsize(BIG)/1e6))
print("  islink : %s   (if True, keep .cache — it holds the real data)" % os.path.islink(BIG))
try:
    sd = torch.load(BIG, map_location="cpu")
    ks = list(sd.keys())
    print("  VALID  : %d tensors" % len(ks))
    print("  sample keys: %s" % ks[:3])
    print("  contains feature extractor: %s"
          % any(("single_clf_core" in k) or ("two_views_clf" in k) for k in ks))
except Exception as e:
    print("  CORRUPT: %s" % str(e)[:120]); raise SystemExit

print("\n"+"="*70); print("2. WRITE PATCHED RUNNER"); print("="*70)
open(os.path.join(REPO,"mv_patched.py"),"w").write(textwrap.dedent('''
    import os, torch
    _rl, _rlsd = torch.load, torch.nn.Module.load_state_dict
    def _load(f,*a,**k):
        if isinstance(f,(str,os.PathLike)) and not os.path.exists(str(f)):
            print("  [patch] skipping missing init file:", os.path.basename(str(f)))
            return {}
        return _rl(f,*a,**k)
    def _lsd(self,sd,strict=True,**k):
        if isinstance(sd,dict) and len(sd)==0: return None
        return _rlsd(self,sd,strict=strict,**k)
    torch.load, torch.nn.Module.load_state_dict = _load, _lsd
    import multi_view_clf_test as M
    M.main()
'''))
print("  wrote mv_patched.py")

print("\n"+"="*70); print("3. SANITY CHECK — MUST PRINT 0.1214"); print("="*70)
r = subprocess.run(["python3","mv_patched.py",
                    "-c","samples/Calc-Test_P_00127_RIGHT_CC.png",
                    "-m","samples/Calc-Test_P_00127_RIGHT_MLO.png"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout)
if r.stderr: print("STDERR (last 1500 chars):\n", r.stderr[-1500:])

1. CHECKPOINT INTEGRITY
  size   : 516.9 MB
  islink : False   (if True, keep .cache — it holds the real data)
  CORRUPT: PytorchStreamReader failed reading zip archive: failed finding central directory


SystemExit: 

/root/miniconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [33]:
%%bash
F="/root/autodl-tmp/multiple-view-main/models/2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt"
echo "--- FILE TYPE ---"
file "$F"
echo; echo "--- SIZE ---"
stat -c '%s bytes' "$F"
echo; echo "--- FIRST 32 BYTES (a torch .pt starts with PK) ---"
head -c 32 "$F" | xxd
echo; echo "--- LAST 32 BYTES (truncated files end mid-data) ---"
tail -c 32 "$F" | xxd
echo; echo "--- IS IT A READABLE ZIP AT ALL? ---"
python3 -c "
import zipfile,sys
p='$F'
try:
    z=zipfile.ZipFile(p); n=z.namelist()
    print('valid zip,',len(n),'entries; first:',n[:3])
except Exception as e:
    print('not a readable zip:',e)
"
echo; echo "--- WHAT THE SERVER SAYS THE SIZE SHOULD BE ---"
curl -sIL "https://hf-mirror.com/dpetrini/cbis-ddsm-efficientnetb3-multiple-view/resolve/main/2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt" | grep -i "content-length\|x-linked-size\|location"

--- FILE TYPE ---
/root/autodl-tmp/multiple-view-main/models/2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt: Zip archive data, at least v0.0 to extract, compression method=store

--- SIZE ---
516949042 bytes

--- FIRST 32 BYTES (a torch .pt starts with PK) ---
00000000: 504b 0304 0000 0808 0000 0000 0000 0000  PK..............
00000010: 0000 0000 0000 0000 0000 3400 2e00 3230  ..........4...20

--- LAST 32 BYTES (truncated files end mid-data) ---
00000000: 0000 0000 0000 0000 0000 0000 0000 0000  ................
00000010: 0000 0000 0000 0000 0000 0000 0000 0000  ................

--- IS IT A READABLE ZIP AT ALL? ---
not a readable zip: File is not a zip file

--- WHAT THE SERVER SAYS THE SIZE SHOULD BE ---
access-control-expose-headers: X-Repo-Commit,X-Request-Id,X-Error-Code,X-Error-Message,X-Total-Count,ETag,Link,Accept-Ranges,Content-Range,X-Linked-Size,X-Linked-ETag,X-Xet-Hash
location: https://us.aws.cdn.hf.co/xet-bridge-us/68f8b8d63f707413be9ffce3/57

In [34]:
%%bash
cd /root/autodl-tmp/multiple-view-main/models
rm -f 2024-10-01*.pt
wget --continue --tries=3 \
 "https://hf-mirror.com/dpetrini/cbis-ddsm-efficientnetb3-multiple-view/resolve/main/2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt"
ls -la *.pt
python3 -c "
import torch
try:
    sd=torch.load('2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt',map_location='cpu')
    print('VALID —',len(sd),'tensors')
except Exception as e: print('CORRUPT:',str(e)[:100])
"

--2026-09-01 12:18:42--  https://hf-mirror.com/dpetrini/cbis-ddsm-efficientnetb3-multiple-view/resolve/main/2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt
Resolving hf-mirror.com (hf-mirror.com)... 160.16.86.14
Connecting to hf-mirror.com (hf-mirror.com)|160.16.86.14|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://us.aws.cdn.hf.co/xet-bridge-us/68f8b8d63f707413be9ffce3/5794a27c45be567e36b90a41556c5f02e6308f52588df6d5c760a4d2461bd9e9?X-Xet-Cas-Uid=62171e3b6a99db28e0b3159d&response-content-disposition=inline%3B+filename*%3DUTF-8%27%272024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt%3B+filename%3D%222024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt%22%3B&user_id=62171e3b6a99db28e0b3159d&Expires=1788239924&Policy=eyJTdGF0ZW1lbnQiOlt7IlJlc291cmNlIjoiaHR0cHM6Ly91cy5hd3MuY2RuLmhmLmNvL3hldC1icmlkZ2UtdXMvNjhmOGI4ZDYzZjcwNzQxM2JlOWZmY2UzLzU3OTRhMjdjNDViZTU2N2UzNmI5MGE0MTU1NmM1

-rw-r--r-- 1 root root 516949042 Sep  1 12:32 2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt
CORRUPT: PytorchStreamReader failed reading zip archive: failed finding central directory


In [35]:
# ══════════════════════════════════════════════════════════════════════════
# CELL 32 — VERIFY THE CHECKPOINT, THEN RUN WITHOUT THE SINGLE-VIEW FILE
# ══════════════════════════════════════════════════════════════════════════
import os, torch, textwrap, subprocess
REPO="/root/autodl-tmp/multiple-view-main"; M=os.path.join(REPO,"models")
BIG=os.path.join(M,"2024-10-01-14h36m_50ep_best_model_AUC_08643_Best_2-Views-CBIS-EFB3-CVUBP.pt")

print("="*70); print("1. CHECKPOINT INTEGRITY"); print("="*70)
print("  size   : %.1f MB" % (os.path.getsize(BIG)/1e6))
print("  islink : %s   (if True, keep .cache — it holds the real data)" % os.path.islink(BIG))
try:
    sd = torch.load(BIG, map_location="cpu")
    ks = list(sd.keys())
    print("  VALID  : %d tensors" % len(ks))
    print("  sample keys: %s" % ks[:3])
    print("  contains feature extractor: %s"
          % any(("single_clf_core" in k) or ("two_views_clf" in k) for k in ks))
except Exception as e:
    print("  CORRUPT: %s" % str(e)[:120]); raise SystemExit

print("\n"+"="*70); print("2. WRITE PATCHED RUNNER"); print("="*70)
open(os.path.join(REPO,"mv_patched.py"),"w").write(textwrap.dedent('''
    import os, torch
    _rl, _rlsd = torch.load, torch.nn.Module.load_state_dict
    def _load(f,*a,**k):
        if isinstance(f,(str,os.PathLike)) and not os.path.exists(str(f)):
            print("  [patch] skipping missing init file:", os.path.basename(str(f)))
            return {}
        return _rl(f,*a,**k)
    def _lsd(self,sd,strict=True,**k):
        if isinstance(sd,dict) and len(sd)==0: return None
        return _rlsd(self,sd,strict=strict,**k)
    torch.load, torch.nn.Module.load_state_dict = _load, _lsd
    import multi_view_clf_test as M
    M.main()
'''))
print("  wrote mv_patched.py")

print("\n"+"="*70); print("3. SANITY CHECK — MUST PRINT 0.1214"); print("="*70)
r = subprocess.run(["python3","mv_patched.py",
                    "-c","samples/Calc-Test_P_00127_RIGHT_CC.png",
                    "-m","samples/Calc-Test_P_00127_RIGHT_MLO.png"],
                   cwd=REPO, capture_output=True, text=True)
print(r.stdout)
if r.stderr: print("STDERR (last 1500 chars):\n", r.stderr[-1500:])

1. CHECKPOINT INTEGRITY
  size   : 516.9 MB
  islink : False   (if True, keep .cache — it holds the real data)
  CORRUPT: PytorchStreamReader failed reading zip archive: failed finding central directory


SystemExit: 

/root/miniconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3707: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
